# B06C — Evaluasi test checkpoint B06B
Enam run per dataset: dua supervised controls B06B dan empat adaptasi feature KD terinspirasi SSID-KD. Tidak ada training, kalibrasi ulang, atau pemilihan threshold. Semua checkpoint sudah dipilih melalui validation. Threshold tetap0,5.

Jalankan Cell1–6 pada runtime GPU baru. Default CBIS-DDSM; ganti DATASET untuk tiga dataset lain. Sumber B06B dibaca tanpa ditulis ulang. Hasil di `b06c_feature_kd_test_v1`. Evaluasi ulang memakai hasil lengkap setelah verifikasi hash; yang belum lengkap dihitung ulang. Audit validation seluruh enam model harus lolos sebelum inferensi test mana pun.

**Batas interpretasi:** satu seed42; test pernah dilihat pada eksperimen sebelumnya, bukan untouched holdout. INbreast memakai BI-RADS1/2 vs5/6, MIAS abnormal-only. Bandingkan KD dengan control B06B; perbandingan terhadap Hinton B05 perlu mencatat perbedaan trainer/RNG dan control. Tidak ada klaim reproduksi SSID-KD asli. Projector hanya digunakan saat training; checkpoint classifier/backbone direstore tanpa projector untuk inferensi.

## Cell 1 — GPU dan Drive

In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import tensorflow as tf
minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=minor:
    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime sebelum update tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])
import tf_keras as keras
from pathlib import Path
import json,hashlib,zipfile,gc,linecache
import numpy as np,pandas as pd
from PIL import Image
from google.colab import drive,files
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
if not gpus:raise RuntimeError('Pilih GPU runtime')
for g in gpus:tf.config.experimental.set_memory_growth(g,True)
keras.mixed_precision.set_global_policy('float32');tf.config.experimental.enable_op_determinism()
drive.mount('/content/drive')
print('TensorFlow:',tf.__version__,'tf-keras:',keras.__version__)

## Cell 2 — Dataset dan referensi laporan yang sudah diperiksa

In [ ]:
EVALUATOR_SOURCE='import os,sys,subprocess,importlib.metadata as metadata\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\']=\'2\'\nimport tensorflow as tf\nminor=\'.\'.join(tf.__version__.split(\'.\')[:2])\ntry: installed=metadata.version(\'tf-keras\')\nexcept metadata.PackageNotFoundError: installed=\'\'\nif \'.\'.join(installed.split(\'.\')[:2])!=minor:\n    if \'tf_keras\' in sys.modules:raise RuntimeError(\'Restart runtime sebelum update tf-keras\')\n    subprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',f\'tf-keras~={minor}.0\'])\nimport tf_keras as keras\nfrom pathlib import Path\nimport json,hashlib,zipfile,gc,linecache\nimport numpy as np,pandas as pd\nfrom PIL import Image\nfrom google.colab import drive,files\nfrom IPython.display import display\ngpus=tf.config.list_physical_devices(\'GPU\')\nif not gpus:raise RuntimeError(\'Pilih GPU runtime\')\nfor g in gpus:tf.config.experimental.set_memory_growth(g,True)\nkeras.mixed_precision.set_global_policy(\'float32\');tf.config.experimental.enable_op_determinism()\ndrive.mount(\'/content/drive\')\nprint(\'TensorFlow:\',tf.__version__,\'tf-keras:\',keras.__version__)\n\nDATASET=\'cbis_ddsm\' # cbis_ddsm | cmmd | inbreast | mias\nEVALUATE_TEST=True\nDOWNLOAD_REPORT=True\nRESEARCH=Path(\'/content/drive/MyDrive/Research\')\nSOURCE=RESEARCH/\'outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1\'/DATASET\nOUT=RESEARCH/\'outputs/benchmark_4datasets/b06c_feature_kd_test_v1\'/DATASET\nREGISTRY_PATH=RESEARCH/\'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json\'\nEXPECTED={\'cbis_ddsm\': {\'inference_helpers.py\': \'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876\', \'protocol.json\': \'0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd\', \'b06b_engine.py\': \'1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487\', \'nasnet_mobile__from__densenet121/seed_42/calibration.json\': \'aeab55fa4d84cee601c6a8260cdf6c1cf5fdce04209a1628d06d41bc38bbe36a\', \'nasnet_mobile__from__densenet121/seed_42/config.json\': \'6d9d750cb753f9d23b63d6904b99bccc5bbc258d2707b3213c3759a468bb181c\', \'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv\': \'2c043d66f6bad232268594d6c2f7b9fcc20f6e85a6bd7dca01753a0bead689d3\', \'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json\': \'c1f99df526d8c09a8e25f19e6131e4186a4bc6407ab5fdca532cef1e3d282909\', \'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json\': \'9c753b24d4b67dc8e5e1e1d5c928a1b03732e680b3516bcb3a394c4eb7bcbd01\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv\': \'4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json\': \'355aaaf73514b5bd248f825608cf9b787ba390fc1a2617c81ce8eaff870417d0\', \'nasnet_mobile__from__densenet121/seed_42/head/history.csv\': \'a4b55d61c0e089a229407a4c453fca8efc30a520404ceb920e3143d015fa82e7\', \'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json\': \'b2d31770ee71f8c3d057d2f181f8a051b577085ed341521b1ce2d80dc175d8bd\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'2c043d66f6bad232268594d6c2f7b9fcc20f6e85a6bd7dca01753a0bead689d3\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json\': \'abae8d02f65c4575bfbecf6f83d3a3d217f0de3738382e1bb261cc149e476c56\', \'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv\': \'130018e3b2fa624b84a1734a050e7096b9d05f3edaa982f299a7e2a869ba5caf\', \'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json\': \'4d53338ba09fa3c570cbe2cdcfdca283d90fc4e74357ac737e67800eb1c960d0\', \'mobilenet_v2__from__densenet121/seed_42/calibration.json\': \'01d3be5f3b2ed72268f3e8f8347a5147b8900f342f834be4e67778854a004980\', \'mobilenet_v2__from__densenet121/seed_42/config.json\': \'a7ae1e4a5dc4bbe525374a1c1f83b6af53158cf0b95722bc95d620be772bdd88\', \'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv\': \'7fde55c146d296fd4c304d70e6e276f728a56e6864ba250cf75f06a029fcfc9c\', \'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json\': \'66cdca96fe1bd5563b8d42435e2267ff3e5d9cf42dd66a6c0678af579b1046f1\', \'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json\': \'53141539528d9d97ef26b3207453adb629c4bba710004dfd2d22c64ee21a58ff\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv\': \'7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json\': \'a4b1e50be42719160485617f3b7f061a86ae96aa2c51cccba8b0712152bdd954\', \'mobilenet_v2__from__densenet121/seed_42/head/history.csv\': \'b5d36e828e8e10829b297967fe711d8a4d9bb745f576189f8c3130b6d07aa6ea\', \'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json\': \'2d3732d2a54a01153a15260353da4b008f2e4524cb132fc66c65559e35634b51\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'7fde55c146d296fd4c304d70e6e276f728a56e6864ba250cf75f06a029fcfc9c\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json\': \'1f53afd0a42ecd1b3269cfb2be70e4c109f73473d8e33a78cbe90ce0942d8c95\', \'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv\': \'0e3736b9efa7f51ca504da43d78579ea169a7bcad0d849cc50bd9d30f93390a6\', \'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json\': \'1c4560021a5d18c2b5baac61297bd71dde0f559f80d8f721d1ab2b652d625a0b\', \'nasnet_mobile__from__vgg16/seed_42/calibration.json\': \'5d9a0f6675ea9462fdc50bef4757d17bd48a290c983f307da767070a67f85909\', \'nasnet_mobile__from__vgg16/seed_42/config.json\': \'7469d9adf43d73cf136ec531f809ee38171b06dfee5740a95bdcfe745180c894\', \'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv\': \'3c821c87753091b41f2f05ecc6f854431e3aa5e94ec7972d60365a5fb22bb141\', \'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json\': \'1f56fc2446a0baab796fdfecaac8fcd75f4480b7d4db91f919f37622c237cc43\', \'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json\': \'f6362ba596085033c517102cdd79d3eb2b7f27d3042e1aaaf32328b024fdff9d\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv\': \'4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json\': \'00e069adbc61e0f7a4f9bc74f417a5800544a7dd3ad58ec600f8d8054f6a84fb\', \'nasnet_mobile__from__vgg16/seed_42/head/history.csv\': \'9e6c05c4afb4d7c4b7852453b9278059bfd9c0c67656ab7b5b6df645c030eb25\', \'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json\': \'2ed3c258699133e461d8d2bc4657f1fc22e05beee1fe953a6d3874df574ffb62\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'3c821c87753091b41f2f05ecc6f854431e3aa5e94ec7972d60365a5fb22bb141\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json\': \'ecbb6a65097b71a30a7d7dbebdddf4e1f29b0d35f150652194e1730d33ce2a35\', \'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv\': \'901f8c82e38f52663261b29dee06ba1bede4f1b673fb74a755cf46e9cd176f90\', \'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json\': \'24b86d46959fbf667a2ee8e6dfa58d9fbcad378be16eeebadace2415e4a4fad5\', \'mobilenet_v2__from__vgg16/seed_42/calibration.json\': \'f8b6c2bda6ed00c17be2326976d71cc78f5530073e9af68c92983c331165280f\', \'mobilenet_v2__from__vgg16/seed_42/config.json\': \'1b7aa521cf538a1813be292d6d149f65d1c16f2a968c5ea14c05dfccba8f45e6\', \'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv\': \'cc43ff07dd9ae1fa95d10b29819b0e0e0d8152a5c58995842b09c342ce8899eb\', \'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json\': \'70b78bafeb9a9288316b9003a0a91c0e5d962fbdd3769956b829de8522cee442\', \'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json\': \'f3570b00cf400922b9397360351d0513bde249bdde4c944784f488f355ce4655\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv\': \'7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json\': \'a0a8585b4f44983adb7336e341f4e4bde3e59726957b5a207311b548a6de7044\', \'mobilenet_v2__from__vgg16/seed_42/head/history.csv\': \'3a17eecb7079486b41b404b6ec5716eac3933bc93c2e6438a1e9eed607c98cb9\', \'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json\': \'298be2ad1d963948fbd89a94f306d95fe31671b5050ad56c10552f916710f75e\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'cc43ff07dd9ae1fa95d10b29819b0e0e0d8152a5c58995842b09c342ce8899eb\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json\': \'23e71af40a45673561ac8031e0ad2cf50b13b10a8da3f0afbcd3e8c736fce2f1\', \'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv\': \'f897fcc44e66a6f54e244314e6999631f46e52c193d3a0fd32b4ab73e7bf9262\', \'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json\': \'3c6f42436fbcb3cd0f122c817ae3a412f158ee744567bf91d21ccb0401d33b00\', \'nasnet_mobile__from__none/seed_42/config.json\': \'01753709d0c91cd76530a21d989576c408ce30c8156d1ea278ceb4ce123fdfe9\', \'nasnet_mobile__from__none/seed_42/validation_predictions.csv\': \'b5d53fd897eed4c4cc4a6fdd146e8bfafeb3f8aa5e9415cca4c165bc7a9c6122\', \'nasnet_mobile__from__none/seed_42/validation_metrics.json\': \'40130dbb93bf9bc09d93160de91b424b21a461e1a28a0d9df0908d1f94ae24b3\', \'nasnet_mobile__from__none/seed_42/COMPLETE.json\': \'d95d943f72162f0d90e09274da4c28d566fc227cbae5fe79cc2cc780aca099a3\', \'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv\': \'4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76\', \'nasnet_mobile__from__none/seed_42/head/validation_metrics.json\': \'3e91b78bd3cc953cf12d3d4bf2e026a52756fd6d1af3f9cea2619f37450564ed\', \'nasnet_mobile__from__none/seed_42/head/history.csv\': \'71d5add792d309e018b2fdef6615d3bdbe04fb027de5b29142439d3a5d196526\', \'nasnet_mobile__from__none/seed_42/head/COMPLETE.json\': \'0239d1e468ef609c4cab378cc209516e2ef0b104501143a8741bffa94320cfee\', \'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv\': \'b5d53fd897eed4c4cc4a6fdd146e8bfafeb3f8aa5e9415cca4c165bc7a9c6122\', \'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json\': \'61382a7c7b1b036645497f6616196ea9f376301963846df39c7a6eeaf687a104\', \'nasnet_mobile__from__none/seed_42/finetune/history.csv\': \'ecdd8942a32d8c2b0d0e8b4cb6f86fea5d6d5594f5e5f37564dc545ff7c13aad\', \'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json\': \'aee3df97befd97f396472e136aa388fd9499b088782f4ee918d79e12ec9212c0\', \'mobilenet_v2__from__none/seed_42/config.json\': \'846e6f14a3971383a95b60f8ffd0665a09c394914d4c43ba2043b359e718853f\', \'mobilenet_v2__from__none/seed_42/validation_predictions.csv\': \'06b422a186fc2bdff1d4582709a120a56fb202607296843d3b3974334474714e\', \'mobilenet_v2__from__none/seed_42/validation_metrics.json\': \'7c46b8c86e2643b098660141a5642926aa4ecc5ac09e97c89e4f0b0a07e34cab\', \'mobilenet_v2__from__none/seed_42/COMPLETE.json\': \'8a8e2438becc486a8247b87c61c5ae471db26a556808c139dd6a04e5aa1b17f4\', \'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv\': \'7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4\', \'mobilenet_v2__from__none/seed_42/head/validation_metrics.json\': \'0f8be37158987146dd8b3bb9a37161960f202444f99f7e2c8892e20a938541a7\', \'mobilenet_v2__from__none/seed_42/head/history.csv\': \'2f3cc03a4f94d79a1fc4c91776139196bd55b893a89286c5ad50eff66a34ff57\', \'mobilenet_v2__from__none/seed_42/head/COMPLETE.json\': \'f17ffe077819f27989c3cbd1e7ea42387c8cea282f7f7283312d0cde2bc9bc4c\', \'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv\': \'06b422a186fc2bdff1d4582709a120a56fb202607296843d3b3974334474714e\', \'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json\': \'398ef4325a9d5c85e9c6ff01f93939139b0ac51b126e4517dde1e8ba47c126ab\', \'mobilenet_v2__from__none/seed_42/finetune/history.csv\': \'2554aaa76d389220e3de9fa9dfb9520c1e2457b547d6b9353bbc71ae0f84f161\', \'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json\': \'bf3964f6c86e9fdead097b18bf09618df9d6a2db951b1407e579be847adff9ac\', \'teacher_cache/vgg16/COMPLETE.json\': \'2cce2a54cafd73b896da5c7e272b7e8461fe92545dbb92c7e8c45420a4c2b653\', \'teacher_cache/densenet121/COMPLETE.json\': \'e91bde95575bc5f32a422f48c08b350b33fb41e4d031e4171fc9ddbf2cbf7b8b\'}, \'mias\': {\'inference_helpers.py\': \'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876\', \'protocol.json\': \'e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc\', \'b06b_engine.py\': \'1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487\', \'nasnet_mobile__from__densenet121/seed_42/calibration.json\': \'170729db3ce33489ad1dc085a9094a7f5758c6df79856da006e219f7969c47c2\', \'nasnet_mobile__from__densenet121/seed_42/config.json\': \'b375d83a08971b0050c45904d8da82bb42e26d07574675dc3dbdc497b5fe49cd\', \'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv\': \'606df8eac9c784a9002e668326d88f82c4eae9fe9a80a9a344175112d0540d1d\', \'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json\': \'f1ff8905957c98a87a28e4e671629606bd8847ed73fb78e36f7a9e04ae41c833\', \'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json\': \'2335864f0f82ab5da5bbe15a2136fa66174c1e15c155fa01baa212dfbfc0a9ab\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv\': \'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json\': \'2c944fedf494d418858f0bca10c7694b554d043c4a86456cd65c5d937e1e118a\', \'nasnet_mobile__from__densenet121/seed_42/head/history.csv\': \'d77aed584e79f188e0e38c6d94bd94a10184cf1c96f06eb6d4f2e617d4efdc96\', \'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json\': \'e75d551c26fc19c2669334dd07f037b4b1631c31a03dfddfd51b5fcd27d56e73\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'606df8eac9c784a9002e668326d88f82c4eae9fe9a80a9a344175112d0540d1d\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json\': \'f84260ff709b5409b327595a0da8d81a1216e5ae03b2be7582586f91ff5105d3\', \'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv\': \'dcc1942e01f411a286385f5ecb5ddd2189beb9ee949bdde375c5d6b411b03b89\', \'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json\': \'94f9b7a87ae3b6eb4ec12ce26b0c51ea6d4391bab144debfc557dc881a0f9d71\', \'mobilenet_v2__from__densenet121/seed_42/calibration.json\': \'e24271150a954dca895f0d3898189c1c8f88d37df8256cf6d19730ada71145da\', \'mobilenet_v2__from__densenet121/seed_42/config.json\': \'3dba5e2d3d551ae65d1895cb79103a6c3f8447fe2c280a716b2844f1b08cc9bd\', \'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv\': \'6fb5fe028980b038243e5806c17e3f525ea2cd2ea00cdf4799f222b3352e6511\', \'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json\': \'9fa759f7f0cacffc0bf780467b137e7319012858b584bc80214e857aeacac252\', \'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json\': \'94de9b54dcb5fd7996419e131c9b339d96a56cc93772caaac6bbcf3f1b4b4856\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv\': \'558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json\': \'21d6213ea31062382940efb9105614043d134416344ec5d34cc0c1ace358e9a9\', \'mobilenet_v2__from__densenet121/seed_42/head/history.csv\': \'be71246b8eeddf1472a995fe7523f2c8c1cfb9df0920c543834dba9b7527834c\', \'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json\': \'d54a7cf29e53624b4a45323646aa48c37d0611032ea4e668ef20322e8ac0604c\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'6fb5fe028980b038243e5806c17e3f525ea2cd2ea00cdf4799f222b3352e6511\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json\': \'d58f315da48b7a1de46af421d83de3301398efc9d4af731b662ce3a2eebb9218\', \'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv\': \'fca2338866a83a97961d14b20ff46fae800eb72aecece79b5a2bc87f3070dd38\', \'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json\': \'39ba5c43b523248f1e558d00145faf9892fa1dc231ab7487c909a80a1351f490\', \'nasnet_mobile__from__vgg16/seed_42/calibration.json\': \'18c39f057a2810a9c35e1333b55d1adcdda929d0ee795f08949df7e3e24e68b1\', \'nasnet_mobile__from__vgg16/seed_42/config.json\': \'b58688b7826ffe20ed077a8e16f726bf91b7fd7c4e4a5991c108de6692935c76\', \'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv\': \'cfe3159308c2bbf6aa1ea972444f79d399d875f6e753eb4390d41236098e48df\', \'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json\': \'5226deca661e354fd1f39b135ea617aba2ddbb3a385fc20a99777f0b3c024b8f\', \'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json\': \'555a10c544641d98f9fb9df011dfd6e19f4d868fc765ae9f8bc7a094173e4e14\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv\': \'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json\': \'938bfb4a4c4b4520ce8fc44af2cb5a7ba23e5f983a10dded1453a3656b321e2e\', \'nasnet_mobile__from__vgg16/seed_42/head/history.csv\': \'600995669ef918995eacd23cb553950a2cac07720e9bb3abb54994ed5ec31f97\', \'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json\': \'030fbcf3ed99c684e6fa9ef91d8c17cedf2017f8fca7e52c451698e428f938aa\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'cfe3159308c2bbf6aa1ea972444f79d399d875f6e753eb4390d41236098e48df\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json\': \'bfb4296c20fa98ee895840d46b1f967f0644d3e7290174776a6264ec5fa11b88\', \'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv\': \'22712690576890a5d2552cf60102b058e848f6ea18eda69f18689c95189292e3\', \'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json\': \'8f9567f277b8ee1f11f1722e5d60fe29c98bef36a6394be161c5c0dd981dd125\', \'mobilenet_v2__from__vgg16/seed_42/calibration.json\': \'e431cff27db5235effcccc877b509cae79fe6b3dc44dc0b3d59af077f3f2c996\', \'mobilenet_v2__from__vgg16/seed_42/config.json\': \'32dca61b64b657870ea470392423a92cdebd0c3535c6d67e3d47faa827dafbab\', \'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv\': \'485fb2170397bd148110c9e4c856f19e2319dbb039800d1baf74a90536e6dd12\', \'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json\': \'af7d25e755232ba9510bd18e45d120b69e79cc2e41ccea64f3700f9e1ac05cb9\', \'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json\': \'df6c10579454987bbf1bc61ee5348647d01faf762297f9719b0bb7cfd3925996\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv\': \'558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json\': \'b34c07a6ee650482e60313227e9572ba1d393212029e7eee30aebf3dc0f4e6df\', \'mobilenet_v2__from__vgg16/seed_42/head/history.csv\': \'70ab34a10362b29c39bcfdf72b6ea19fe90a333590341a8df50bbf5dc713e7fb\', \'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json\': \'a5465c4e888c77fae824b4e5c3b4ea877f4cbb931e42dde525660f5eca9ac007\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'485fb2170397bd148110c9e4c856f19e2319dbb039800d1baf74a90536e6dd12\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json\': \'4252a862aba8186176c310392747b7ab7e808c78bfc0df6c8e0280119d2086d1\', \'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv\': \'1d4cbadaf1797bdb21c1f2ec289c4d37c48423387e16100027e5f72faab8b0ed\', \'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json\': \'02b9ef78ce26b0cdb0ec37de1597a25c1463f9c8fa0fbd5d0d3804bc458c919b\', \'nasnet_mobile__from__none/seed_42/config.json\': \'3cff9a8084913b5abf8fae84b19f11b8009d2c124deda0a5dc2e0e5ffeecb8e9\', \'nasnet_mobile__from__none/seed_42/validation_predictions.csv\': \'5bfdcf278205b7222a6f534974fc4e402e36b09e4f4631f5d1a5b55a5fe34263\', \'nasnet_mobile__from__none/seed_42/validation_metrics.json\': \'0f48694d80b21c3c74a3971b68395e3e264ac7f264df6463fb3ecb83074dc964\', \'nasnet_mobile__from__none/seed_42/COMPLETE.json\': \'48802486ae91b6336cb8e835c871259deaa7d3819f983cd7a62d68f364298bb1\', \'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv\': \'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113\', \'nasnet_mobile__from__none/seed_42/head/validation_metrics.json\': \'6f9f218a0ddac8d0e84c731506a9a62d0ba5905163bf3b1e4c996703297e6f98\', \'nasnet_mobile__from__none/seed_42/head/history.csv\': \'58e0e1aac4f9e46d956a8af1be921fcb492e3e5838e4ba26b82caa4636ca9078\', \'nasnet_mobile__from__none/seed_42/head/COMPLETE.json\': \'6f3b2f3f50e47f32dccadafb5a74afb41e6e4c52dc8aac462df3bd55cbf1e464\', \'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv\': \'5bfdcf278205b7222a6f534974fc4e402e36b09e4f4631f5d1a5b55a5fe34263\', \'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json\': \'368737cef28eaaeba33d11db11c62fe9652fc571cb39282631dae44a8334f4d2\', \'nasnet_mobile__from__none/seed_42/finetune/history.csv\': \'a5bac70b6cb4a49a2b089183b572122ebc3ec578ff1d29cf53da6b05b5bd3bf9\', \'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json\': \'6e80b4c27584fc04e33ebea2f42eff92fd4d3fc225a63d544fe0b5e4103ea120\', \'mobilenet_v2__from__none/seed_42/config.json\': \'0ac93d71ce607da5b43a11a05a2553c758fb19bb1e63ad5f097f6c4a5fe11fc4\', \'mobilenet_v2__from__none/seed_42/validation_predictions.csv\': \'2447eb121c86694080a7c1d824a66350fc850d5cd35f8482ecf4e81814999538\', \'mobilenet_v2__from__none/seed_42/validation_metrics.json\': \'cc9b05219bfe09e39a31544c6063286842cbcf88113f0ce7bfa6698f1c375c99\', \'mobilenet_v2__from__none/seed_42/COMPLETE.json\': \'e6b1988b8d48ff44709227cde1c482b47660911c5d8bffa590da51aafdc932cc\', \'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv\': \'558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044\', \'mobilenet_v2__from__none/seed_42/head/validation_metrics.json\': \'c7735a30e5d72ad75c580d926ae751737f40ad31a7f0cc63942f3ed76fb22502\', \'mobilenet_v2__from__none/seed_42/head/history.csv\': \'22bc8564cbaa6d075e445bd893c0426925f488be63a667490a3852f3296643b3\', \'mobilenet_v2__from__none/seed_42/head/COMPLETE.json\': \'1c49e26b935b4fdc0b73fae04d8158740dab5fffe7e43302c5e0873d318574b3\', \'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv\': \'2447eb121c86694080a7c1d824a66350fc850d5cd35f8482ecf4e81814999538\', \'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json\': \'4d918e7c1af0b8cb3ce7c899e116060b20c596a409119144d87804fd978e9196\', \'mobilenet_v2__from__none/seed_42/finetune/history.csv\': \'0a3001ffab045aada980d973cdb94a746ef291f547820e51d374ab4b51bd452f\', \'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json\': \'0bec00534ac3cc6a7bdebef15b729c1084d1ca297b36284664658c412f602e6e\', \'teacher_cache/vgg16/COMPLETE.json\': \'d7e9cb02aa9c03918d665234b7ad77dc12964907df7b1133b96517a7b1d449fb\', \'teacher_cache/densenet121/COMPLETE.json\': \'1c6e3afc06faa08b88b62825e84992f89c29ce8570b2033cfa38e84f2794242c\'}, \'cmmd\': {\'inference_helpers.py\': \'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876\', \'protocol.json\': \'994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77\', \'b06b_engine.py\': \'1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487\', \'nasnet_mobile__from__densenet121/seed_42/calibration.json\': \'40ee044aef90bd7b86c7578249d45a518d32f32704fe475ab934532a1765377b\', \'nasnet_mobile__from__densenet121/seed_42/config.json\': \'015fab1a4226c90d7ec799263ec9e30daa9d20fbaefca2edf1c3381d30d7e808\', \'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv\': \'7c6edf398625de8fd0d027b2b13c223190817e3ef54ee3092f466b87b6e92e86\', \'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json\': \'951051d9634df0da67e50dba50023ca134ea78f2df1b9608d2917137500ed184\', \'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json\': \'c69d02685b3c342da6c7ab699f12f1b8ffd51befe6918c8cb3ce8ceff5dcab1b\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv\': \'12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json\': \'8897110e2edbddd3ff1b7a55422b8daf77a7dcb622c4301cc1907c55fd9f4b1b\', \'nasnet_mobile__from__densenet121/seed_42/head/history.csv\': \'ae35e41635088d64e00513fb304f4fc71438c055a25eedd2e3fdc1f3e35df6f6\', \'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json\': \'89e288222f79e468797e9ce1264458e0f6570009eb7db80751f54798abf6d6e3\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'7c6edf398625de8fd0d027b2b13c223190817e3ef54ee3092f466b87b6e92e86\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json\': \'12700491247aa7e7933f1b8b311afb9cff4f80e89d3f5637e5d520b24931151d\', \'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv\': \'2b45c733ecf4a2c6830b4dd0b72e17754a58cfdb5df6e08c6092a6f940c6f407\', \'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json\': \'01e7b8e24719d73db2d2c5df535fbfbc039f5bd52d2df126eee91f1fba7359e9\', \'mobilenet_v2__from__densenet121/seed_42/calibration.json\': \'9b397ca23b3ddbafa86d981ebc53f1a35b2a2c8ef3ba2cf9f1045fbcaf7a235d\', \'mobilenet_v2__from__densenet121/seed_42/config.json\': \'84d5ea3ce34c1cd57fede737dcab15eca154d3e513265bdff78b3ebed7036c9a\', \'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv\': \'975a2ea6bb724c05fc0c2713e1f6788bc80d33a256ddb3b4e06f045c80dd398c\', \'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json\': \'e5027aad97bff50f13d5e8e57fef7a01fa6782fb0220ab7c18df01c800351f1d\', \'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json\': \'c3c9db59e25b40b54fbc86b1aa614975f2279a56c840af833bf596784f3e14e8\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv\': \'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json\': \'88e1358a2e8d153752614b00bf35192b80b000e483df4d1d925e018ea0944be0\', \'mobilenet_v2__from__densenet121/seed_42/head/history.csv\': \'b9dd2da2c4cba2a8b40154e87b9471758eb252089876387142c9bedf3abfc53c\', \'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json\': \'881c6aa28b1f0972d73ddc301aff889e943491dac6942c795f2e2f2ba151d5d3\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'975a2ea6bb724c05fc0c2713e1f6788bc80d33a256ddb3b4e06f045c80dd398c\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json\': \'f2b43de1dd37de927585f9d925af9cf56a5c0808aafda52a3aad2287ea421ada\', \'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv\': \'fd6b8236f2d98b31b3348097e488fa7999d7f34a9b86d52439c353017b877ca2\', \'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json\': \'fd416cfba65a87338fdc07315d843c63ba4c8c4163f472b62d6b3c095330972e\', \'nasnet_mobile__from__vgg16/seed_42/calibration.json\': \'7374826c8afcf59f262fd1f29901e3176baf60581b9ac80aa0c247b350954a83\', \'nasnet_mobile__from__vgg16/seed_42/config.json\': \'ceb47867c479168c8193c21c6914b7a04dc5f1c7beeabacd15203bb4d4ba4bad\', \'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv\': \'6287d0782cc656314b5c7f3acbe9b71717fc25812441eaafb9390739e85ed5da\', \'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json\': \'cde0d42ab9bab32fac7977603320e89d23405b5b8d36cccab0c8bf4bc4e6d1a7\', \'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json\': \'bf22344b0319f5eb0525babfee8fc824a16b8a5c474e075462d1c31e34c7070c\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv\': \'12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json\': \'810bf5d63d356dbdb85d61a47a50cb13e2805310a07349e56957667bf6610c7d\', \'nasnet_mobile__from__vgg16/seed_42/head/history.csv\': \'e1ee3fd61193503cd14cd7a35b94a42bcdf9a224243dbc6f55a7e829f4050fb1\', \'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json\': \'8c4d7d83f385b2f16a1a959c3bb0377644845180b2f61b977c8b1c00f16b0189\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'6287d0782cc656314b5c7f3acbe9b71717fc25812441eaafb9390739e85ed5da\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json\': \'2dac829e61099783bbcde8a786a597c23128c2c64392eea776d35450022ed846\', \'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv\': \'7c94380d04a184e2ffb610ce55a6ce6dee23380e1176e67186e0a4988f825da0\', \'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json\': \'4f0fdce1dc2d5ff5ed6d1c99beaa960fdf9cd4d51ef18defa2a9a62273e8a03c\', \'mobilenet_v2__from__vgg16/seed_42/calibration.json\': \'38a4ffbbbbcd026e91a91a7bed61235f93e607ec90c867f71f49459a44668d85\', \'mobilenet_v2__from__vgg16/seed_42/config.json\': \'670f81e29a287b8192fe2386dcf32088c63facfd4e7cc518c02170997c6e9f68\', \'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv\': \'e7c7a425cd0d0704e9626c7a810349d12835b8ba2079fa17497c4fd439340e81\', \'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json\': \'67c4cdf07a5d9148819d5d5843f39c8a66dd102c1891fca3c33a695674c231f1\', \'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json\': \'a65f9476970326e198b95381af507b93a1ff43470fd2e207cb0f0f11b5468694\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv\': \'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json\': \'5f0b0b34b5690e0c1a8706733cdc600061799d1ac9a5f85e446a06ad411f54a8\', \'mobilenet_v2__from__vgg16/seed_42/head/history.csv\': \'e42bca4ff948735df98b02a474c39a090a9e7a2de53b56f2a782cdc8f54655a9\', \'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json\': \'0273387406c8d8ba37ca1925fd98b47ca1e2111f3039385afe3ba9d7df25fd60\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'e7c7a425cd0d0704e9626c7a810349d12835b8ba2079fa17497c4fd439340e81\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json\': \'ff4cc92399471f8919ac0fb6f5f576ed357266e574cf6e155596fbad63f1f3ea\', \'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv\': \'039db4de369960bcf3d5b2ad9a716e1a065950e22723444faa2a9f0c71fe13ce\', \'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json\': \'ae20cd272d93e895caa67263c13bab879702b341e53194608b6f9504a03b20d6\', \'nasnet_mobile__from__none/seed_42/config.json\': \'84fb9053e6df544e7682c3740e231d9a0d2d9f60d97f14be96ec8e90a3abf870\', \'nasnet_mobile__from__none/seed_42/validation_predictions.csv\': \'78947db308cfbdd255341993052107d1b389a72f3c32f33c41971acbaf954ebf\', \'nasnet_mobile__from__none/seed_42/validation_metrics.json\': \'9e85707f820b2badc0ce94a144563f91faaa7fe808357a4db3e2970cd58b707d\', \'nasnet_mobile__from__none/seed_42/COMPLETE.json\': \'f54d0ef9209d01def6d538a489161b785f86743f56c414b864344aaefad6a2e2\', \'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv\': \'12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df\', \'nasnet_mobile__from__none/seed_42/head/validation_metrics.json\': \'88a87226154ab3434a45df2845b0631fe5a575646d25be3c24727d009558a752\', \'nasnet_mobile__from__none/seed_42/head/history.csv\': \'65d31a3de51f91fbd59b2f934a8eae28787bfc296356b2e1d14e1a3db40cb35e\', \'nasnet_mobile__from__none/seed_42/head/COMPLETE.json\': \'97798d8d14076613a76f072e1a1f21e1862e6d0e316412412baad333e398ecee\', \'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv\': \'78947db308cfbdd255341993052107d1b389a72f3c32f33c41971acbaf954ebf\', \'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json\': \'031e59683dbfe0e2a70dd5597adb24f2db07870cfc188946b27cce397327fbf5\', \'nasnet_mobile__from__none/seed_42/finetune/history.csv\': \'6879a5c22c41410134359315c71ee14bc43fa94e6d6946e323c9e21077a06706\', \'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json\': \'b5b8c56a7faac7281f1ae5e8373092156e1b2f9dff89b06a82388b1bb050c6ef\', \'mobilenet_v2__from__none/seed_42/config.json\': \'0025a1bee6e8cd0cbfd41d09fbb03cead63eb311b52fb223654e59fd3e62adb5\', \'mobilenet_v2__from__none/seed_42/validation_predictions.csv\': \'a0c26ee58a67f0aa2b5d24a2a02bdcc10ecfbb15832a8c484c2e3263254e9543\', \'mobilenet_v2__from__none/seed_42/validation_metrics.json\': \'a6e4815c2b418a187fceedc1f1c7c2c33366a843d2890216d47892b35d281ebb\', \'mobilenet_v2__from__none/seed_42/COMPLETE.json\': \'74d9ad8f85b27faaa5807429a4f2d4ca72c3acac8fd44abfad828ed8321255dd\', \'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv\': \'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe\', \'mobilenet_v2__from__none/seed_42/head/validation_metrics.json\': \'42af56c0726630beda2eb37179231097c955ab81ba96f3294a5eb532272ead0f\', \'mobilenet_v2__from__none/seed_42/head/history.csv\': \'5fcee53f341317fa8e976557906a81f74d0a130135a55951a28bc1a36a1a6d83\', \'mobilenet_v2__from__none/seed_42/head/COMPLETE.json\': \'e64247e6773fd8b53a6bad3ac57c0599a63ae4840f2366ab8c669618161be6ac\', \'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv\': \'a0c26ee58a67f0aa2b5d24a2a02bdcc10ecfbb15832a8c484c2e3263254e9543\', \'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json\': \'ab4e79405b3ddc4c9b13466c81c887da1707437dcd7e8be7967fc540dee0312a\', \'mobilenet_v2__from__none/seed_42/finetune/history.csv\': \'7daa72fcb212173b8f2617c4ef768c34c5eab78e2aac0820c67fcc7948ed638f\', \'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json\': \'06bef1a97669990c3c6906b05c7278278e764e5b3cf1ace79cd7a50a2481f532\', \'teacher_cache/vgg16/COMPLETE.json\': \'7fadb5a4d8696c83925ca06a9b3cc4dce668bfd1254060e196b089f8ddca8fce\', \'teacher_cache/densenet121/COMPLETE.json\': \'413897bd9db8d85d31302d702a9c5073d35076635b7aaf9f513ca1c7c36faa12\'}, \'inbreast\': {\'inference_helpers.py\': \'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876\', \'protocol.json\': \'fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988\', \'b06b_engine.py\': \'1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487\', \'nasnet_mobile__from__densenet121/seed_42/calibration.json\': \'7c2485859a7045ec7b3b2d441d1286d0a78c0d5aee96797783b240c6e0f4970e\', \'nasnet_mobile__from__densenet121/seed_42/config.json\': \'1a2f5ca656439b3231965b73dc7ee1448da81bbc627bb47af44612bfc2a9ca2e\', \'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv\': \'215eab0f1f3d35b1b82121539cb72c7f7f87087a0da5722c0d809478af4bb61f\', \'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json\': \'fe3a3f44855b16f492b73d78192de912e6da84460ced8a82928dbb7c12750b04\', \'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json\': \'fe4a51a734853ca7aa964ef0c5a053c4998beceb8581f2988027414cc70c539a\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv\': \'1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29\', \'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json\': \'bd76761a49fad9cd03a09ba793199cbc9f68da0046b78e6b3528c7fa2ebb010e\', \'nasnet_mobile__from__densenet121/seed_42/head/history.csv\': \'f9847c23987517b4720285dd557d223cbf5d0d206e55c3e8b5a2ae5629727b2c\', \'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json\': \'938ac84e64725003dd648ce334d547c063bcd6b64dbdd8a04ceff91bb83be605\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'215eab0f1f3d35b1b82121539cb72c7f7f87087a0da5722c0d809478af4bb61f\', \'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json\': \'5d3c99f2f092b1141e6efc6a6a7457b94ebb4771ee8451a1afe30c7afea2d9fe\', \'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv\': \'db8cd48d98ff18cdfc9ec74396cb80ee18e97d365f2e852ef1e412f19e21c337\', \'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json\': \'1dfe2f0eecbf4123f9ce5b962f3286fbaea7b509e5a724e4369fd744e808ac7c\', \'mobilenet_v2__from__densenet121/seed_42/calibration.json\': \'70633ab6b36020e024f53ce362fccfd4f44c9c5abc1be2a6bfd65bcaaba7fce4\', \'mobilenet_v2__from__densenet121/seed_42/config.json\': \'eb1da87bbcf14145a9b9714a39e0cfa1c5054da06a438ebfe431add8ea35142a\', \'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv\': \'ea3bc1497a53528a25fd205ef4d98b7bbb3b9610716f5f7877571b960ec3b581\', \'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json\': \'e929c2d60a36f0390e7ac8e455d01e3df125fd776b92eb0505048ce4cd6ae75b\', \'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json\': \'0fe44cd5840c1e62ae8c3283f002814dc6ea90c9613042ca54d9b42adadd783f\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv\': \'761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197\', \'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json\': \'54af2389f78cda620aef1f19e8bf2b51ba8d55620b1944776bf5dfabc15168f1\', \'mobilenet_v2__from__densenet121/seed_42/head/history.csv\': \'3ce49778bf38cfd4f8142d101b53956b8acadaca6ebfe2149b8a1841c4749865\', \'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json\': \'7492a3da9b994a237676240b90e39b183843e3fb76b7fd05aa1f19113556674b\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv\': \'ea3bc1497a53528a25fd205ef4d98b7bbb3b9610716f5f7877571b960ec3b581\', \'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json\': \'20fcc37db1fe1f703cca24ce5b4ce1106490ece0d8734343091be2bf1d066408\', \'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv\': \'c623dac1f026bd2f1a706e6dbf132e9ff44a3c2c138eb368b7e101a3b53de980\', \'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json\': \'963175d0c6e0c8d784031657118bfe8a150adce56d3334783fcd380d9149f51f\', \'nasnet_mobile__from__vgg16/seed_42/calibration.json\': \'e8901cedda5acd7f222515239a221a250c5029959bfdd70d8a85737368b84258\', \'nasnet_mobile__from__vgg16/seed_42/config.json\': \'fa4d39634389a195a977108b4266bc00c1a5850deb7b3e470c383f3ddf71c121\', \'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv\': \'594141687dcaa99c1c016165d0938e32843980c0caafbcf8d0222779f6f0732d\', \'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json\': \'b0452db65a61f64c42cf9e105b42920cfe89ef6842d4627260bf3010e530a1a9\', \'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json\': \'f32ac38b692da60dbfebde3763da432205aee39f38e08b3033b0e8de411e36aa\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv\': \'1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29\', \'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json\': \'cd1e45def4f364adfe4c2bd5195f27a1cd8a404d7dccee857975ddfb50274310\', \'nasnet_mobile__from__vgg16/seed_42/head/history.csv\': \'da3e2d90a8e16e895e6ac7d03fbdd510d5d09b3c40b44f0e60c0cbf7b685d409\', \'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json\': \'b2df91949e02c87fb45b5a0c377ed2f1969c73a305e57650b1d77b8be4d01791\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'594141687dcaa99c1c016165d0938e32843980c0caafbcf8d0222779f6f0732d\', \'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json\': \'cd5d225d04e937abf29b2ef4a29a18c0a9db0ee52f6766b5e1afc320fcd36d72\', \'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv\': \'ebb3c81b8cc089806765624a541d878dfcf05b023d6a0667192053cc77d780f4\', \'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json\': \'feb2e8a4a57b74c859a6484f1be316a46ea17b869504ab12a02cf614b1713daf\', \'mobilenet_v2__from__vgg16/seed_42/calibration.json\': \'18235f0d15caa10ed6fba88eaffc771b2688960eaf24cffea9ed0512b5ebb2fb\', \'mobilenet_v2__from__vgg16/seed_42/config.json\': \'7fd5f8e90cd65f0e5ab5f01fe909729a65cc8190da905e0fab4da4b879eec7c0\', \'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv\': \'4fd490f53970e20659a95841ceb44168842e305740b612a481dfb6c2f4829e67\', \'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json\': \'0ec2fd2a1c6a03ab46825ba0a97e7d40b8f6688faced966eb01ecae724187527\', \'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json\': \'75c248e2a342949fe45b0fa446db3d4ed317397f62d4ed4301d9a49ce108a3cd\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv\': \'761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197\', \'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json\': \'731e9cbd09caf95d0a57f455011aad41d160ad09f88f0091b0094c1405257f0a\', \'mobilenet_v2__from__vgg16/seed_42/head/history.csv\': \'0863711ced936eeae3ffede6b4f0370de2d4d51436dae51273f95d93d84632a3\', \'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json\': \'b4ad183401803738f2140c5e974cacfc1deace3c08499138f649ddde9978fae2\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv\': \'4fd490f53970e20659a95841ceb44168842e305740b612a481dfb6c2f4829e67\', \'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json\': \'0c140234e242032d2b09f4dcfb9f56dfd285d3170416b428a0b670cd3a80b041\', \'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv\': \'ebec73e6fe896c9c4abeed46dc9d7f6719cd14c1804675b051061141edfd1412\', \'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json\': \'7e67c01093207c446709026819208605646b76cd6a23087eba3605b42b0f110a\', \'nasnet_mobile__from__none/seed_42/config.json\': \'30d1a8d844f5987fc1bf12edc0a12891928608e45627b8092eea7dce3897fc07\', \'nasnet_mobile__from__none/seed_42/validation_predictions.csv\': \'4c7df2289f8da997b65ad874aec78e25eb4cc9111c5022b69800061b9ad4f1a2\', \'nasnet_mobile__from__none/seed_42/validation_metrics.json\': \'cd4064bf792618234aa243ca4efc830d577cacade26a30fd671ff2cb51114179\', \'nasnet_mobile__from__none/seed_42/COMPLETE.json\': \'e04056fbc448366c9daf5081742bf4e8f725c4a59f78dff4c936fac45267e98c\', \'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv\': \'1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29\', \'nasnet_mobile__from__none/seed_42/head/validation_metrics.json\': \'54fa1f99208e7a2acaafff7dc256778e6fee03c323019f4869654a1d9c7db0ee\', \'nasnet_mobile__from__none/seed_42/head/history.csv\': \'ed655cca3e4164110acdbbf51f44cbeee86c1cdaced39c8d8358d9f778b00fa3\', \'nasnet_mobile__from__none/seed_42/head/COMPLETE.json\': \'06ea3c4da1501d01e96aaaf999acee615684c7957284480c18ba8c707f53b475\', \'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv\': \'4c7df2289f8da997b65ad874aec78e25eb4cc9111c5022b69800061b9ad4f1a2\', \'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json\': \'ea1e85afd3f288a4ed8479dbf82f41abaeab1050ad1642dd5cb38b7972d9403c\', \'nasnet_mobile__from__none/seed_42/finetune/history.csv\': \'4e46a32c16b9ea30b4405160289ef06654c9c054e5fdb5e2ae9333ccac966896\', \'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json\': \'3e40d7aad9cf614d7d4250d85724c5c5ee9e003e82d6389ca5e42df1d16196ea\', \'mobilenet_v2__from__none/seed_42/config.json\': \'480b25a6406d8037fe5a0e74785c70d56d1cce89d7109ddd19d79a97ffa8743c\', \'mobilenet_v2__from__none/seed_42/validation_predictions.csv\': \'41054a104fee39c7609029b470c9dc805413b80593b8d102fedf7e4188f87895\', \'mobilenet_v2__from__none/seed_42/validation_metrics.json\': \'68f4bf60db27ba0dc8e48cf83a9d8de7c62d718a8c096c741242ea9c7859f573\', \'mobilenet_v2__from__none/seed_42/COMPLETE.json\': \'858f64b58128a4f54f461b55548f2cc13068a504d02a3ebcf3657fd659179a9e\', \'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv\': \'761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197\', \'mobilenet_v2__from__none/seed_42/head/validation_metrics.json\': \'49178d1f51a961fa64153d8e9e4ae1f6891d7744ff2844f5c77e5c14deb2e4cb\', \'mobilenet_v2__from__none/seed_42/head/history.csv\': \'411b8cf1be9cc4104313d75149eb957b54199f593f686ec25f196dc92493fc2e\', \'mobilenet_v2__from__none/seed_42/head/COMPLETE.json\': \'e1ca734d4b1b65a6d3150a8273e654195c5ef570c35ff350774b8895b81ba8d0\', \'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv\': \'41054a104fee39c7609029b470c9dc805413b80593b8d102fedf7e4188f87895\', \'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json\': \'1f2ec3ed235830a847529554351695c30563aa277eb8549a4d95d41b44b5f975\', \'mobilenet_v2__from__none/seed_42/finetune/history.csv\': \'a89a2490311368d27771fa477706912bda982b01af8b876e47ec6e7f62e59c17\', \'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json\': \'bccec9143a64e81c40137387663ed2915efe4ed0482aa602d676b3f82ec30282\', \'teacher_cache/vgg16/COMPLETE.json\': \'e42f62737f69a7560e1e9f2dd6a200064b4b04f02c5f945bfd050cd889ada97e\', \'teacher_cache/densenet121/COMPLETE.json\': \'428da9889dfa70c5feabc524e32da88f9e0ce552c0a7a8a53429399002ae2980\'}}\nif DATASET not in EXPECTED:raise ValueError(\'Unknown dataset\')\nif SOURCE.resolve()==OUT.resolve() or SOURCE.resolve() in OUT.resolve().parents:raise RuntimeError(\'Output overlaps training\')\ndef read_json(p):return json.loads(Path(p).read_text())\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\ndef objsha(o):return hashlib.sha256(json.dumps(o,sort_keys=True).encode()).hexdigest()\ndef save(p,o):\n    p=Path(p);tmp=p.with_suffix(p.suffix+\'.tmp\');tmp.write_text(json.dumps(o,indent=2,allow_nan=False));os.replace(tmp,p)\ndef guard(p,o):\n    if Path(p).exists() and read_json(p)!=o:raise RuntimeError(\'Existing lock differs: \'+str(p))\n    save(p,o)\nSOURCE_HASHES={}\nfor name,digest in EXPECTED[DATASET].items():\n    p=SOURCE/name\n    if sha(p)!=digest:raise RuntimeError(\'B06B differs from reviewed report: \'+name)\n    SOURCE_HASHES[str(p)]=digest\nPROTOCOL=read_json(SOURCE/\'protocol.json\')\nhelper=(SOURCE/\'inference_helpers.py\').read_text()\nhelper_path=\'/content/b06c_inference_helpers.py\'\nPath(helper_path).write_text(helper)\nlinecache.cache[helper_path]=(len(helper),None,helper.splitlines(True),helper_path)\nexec(compile(helper,helper_path,\'exec\'),globals())\n# Helper only defines models, data transforms and metrics; no training entry point.\nRUNS=[]\nfor teacher in [\'none\',\'vgg16\',\'densenet121\']:\n    for student in [\'mobilenet_v2\',\'nasnet_mobile\']:\n        folder=SOURCE/(student+\'__from__\'+teacher)/\'seed_42\'\n        cfg=read_json(folder/\'config.json\');done=read_json(folder/\'COMPLETE.json\')\n        if done[\'config_sha256\']!=sha(folder/\'config.json\') or done[\'epochs_completed\']!=30:raise RuntimeError(\'Incomplete/config mismatch\')\n        if cfg[\'student\']!=student or cfg[\'teacher\']!=teacher:raise RuntimeError(\'Pair mismatch\')\n        for k,v in PROTOCOL.items():\n            if cfg.get(k)!=v:raise RuntimeError(\'Run protocol mismatch: \'+k)\n        prefix=Path(done[\'best_checkpoint\'])\n        if folder.resolve() not in prefix.resolve().parents:raise RuntimeError(\'Checkpoint outside run\')\n        parts=sorted(prefix.parent.glob(prefix.name+\'.*\'))\n        if not Path(str(prefix)+\'.index\').exists() or not any(\'.data-\' in p.name for p in parts):raise FileNotFoundError(\'Missing checkpoint \'+str(prefix))\n        for p in parts:SOURCE_HASHES[str(p)]=sha(p)\n        RUNS.append(dict(student=student,teacher=teacher,folder=folder,checkpoint=str(prefix),config=cfg))\nprint(\'Six completed runs found. No training invoked.\')\n\nentry=read_json(REGISTRY_PATH)[DATASET][\'full_512\']\nif entry[\'status\']!=\'READY\':raise RuntimeError(\'Cache not ready\')\nfolder=Path(entry[\'path\']);meta=read_json(folder/\'READY.json\')\nif meta[\'dataset\']!=DATASET or meta[\'image_size\']!=512 or meta[\'file_sha256\']!=PROTOCOL[\'data_sha256\']:raise RuntimeError(\'Data differs from B06B\')\nFRAMES={}\nfor split in [\'train\',\'validation\',\'test\']:\n    path=folder/(split+\'.csv\')\n    if sha(path)!=PROTOCOL[\'data_sha256\'][path.name]:raise RuntimeError(\'CSV changed\')\n    f=pd.read_csv(path,dtype={\'sample_id\':str,\'patient_id\':str})\n    if f[[\'sample_id\',\'patient_id\',\'label\']].isna().any().any() or f.sample_id.duplicated().any():raise RuntimeError(\'Invalid identities\')\n    if set(f.label)!={0,1} or set(f.split)!={split}:raise RuntimeError(\'Invalid labels/split\')\n    FRAMES[split]=f\nallf=pd.concat(FRAMES.values(),ignore_index=True)\nif allf.sample_id.duplicated().any() or (allf.groupby(\'patient_id\').split.nunique()>1).any():raise RuntimeError(\'Split overlap\')\nif sha(folder/\'images.zip\')!=PROTOCOL[\'data_sha256\'][\'images.zip\']:raise RuntimeError(\'Archive changed\')\nIMAGE_DIR=Path(\'/content/b06c_images\')/DATASET/PROTOCOL[\'data_sha256\'][\'images.zip\'][:16]\nIMAGE_DIR.mkdir(parents=True,exist_ok=True)\nwith zipfile.ZipFile(folder/\'images.zip\') as archive:\n    for row in pd.concat([FRAMES[\'validation\'],FRAMES[\'test\']]).itertuples():\n        member=row.cache_member\n        if Path(member).name!=member:raise RuntimeError(\'Unsafe cache member\')\n        dest=IMAGE_DIR/member\n        if not dest.exists():dest.write_bytes(archive.read(member))\n        with Image.open(dest) as im:a=np.asarray(im)\n        if a.shape!=(512,512) or hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:raise RuntimeError(\'Pixel mismatch\')\nLOCK=dict(version=\'b06c_v1\',dataset=DATASET,seed=42,batch=8,threshold=.5,\n    source_hashes=SOURCE_HASHES,data_sha256=PROTOCOL[\'data_sha256\'],tensorflow=tf.__version__,tf_keras=keras.__version__,\n    restore_atol=1e-5,restore_rtol=1e-5,test_status=\'Previously inspected in earlier experiments\',\n    evaluator_sha256=EVALUATOR_SHA256)\nOUT.mkdir(parents=True,exist_ok=True);guard(OUT/\'evaluation_lock.json\',LOCK)\nsave(OUT/\'training_protocol_reference.json\',PROTOCOL)\nAUDIT=pd.DataFrame([dict(split=s,images=len(f),patients=f.patient_id.nunique(),negative=int((f.label==0).sum()),positive=int((f.label==1).sum())) for s,f in FRAMES.items()])\nAUDIT.to_csv(OUT/\'split_audit.csv\',index=False);display(AUDIT)\ndef unchanged():\n    for p,h in SOURCE_HASHES.items():\n        if sha(p)!=h:raise RuntimeError(\'Source changed after lock: \'+p)\ndef align(saved,ref):\n    if saved.sample_id.duplicated().any() or set(saved.sample_id)!=set(ref.sample_id):raise RuntimeError(\'Prediction sample IDs mismatch\')\n    saved=saved.set_index(\'sample_id\').loc[ref.sample_id].reset_index()\n    for c in [\'sample_id\',\'patient_id\',\'label\']:\n        if not np.array_equal(saved[c].to_numpy(),ref[c].to_numpy()):raise RuntimeError(\'Identity/label mismatch \'+c)\n    return saved\nprint(\'Output:\',OUT)\n\ndef restore(run):\n    keras.backend.clear_session();gc.collect()\n    m=build_model(run[\'student\'],512,42,weights=None,stage=\'head\')\n    status=weight_snapshot(m).restore(run[\'checkpoint\'])\n    status.assert_existing_objects_matched()\n    status.expect_partial() # checkpoint contains training-only projector\n    return m\ndef predict(m,run,split):\n    return m.predict(make_dataset(FRAMES[split],IMAGE_DIR,run[\'student\'],8,42,size=512),verbose=0).reshape(-1)\nRESTORE_PASSED=False;rows=[]\nunchanged()\nfor run in RUNS:\n    m=restore(run)\n    saved=align(pd.read_csv(run[\'folder\']/\'validation_predictions.csv\',dtype={\'sample_id\':str,\'patient_id\':str},float_precision=\'round_trip\'),FRAMES[\'validation\'])\n    p=predict(m,run,\'validation\');q=saved.probability.to_numpy(float)\n    passed=bool(np.isfinite(p).all() and np.allclose(p,q,atol=1e-5,rtol=1e-5) and np.array_equal(p>=.5,q>=.5))\n    rows.append(dict(student=run[\'student\'],teacher=run[\'teacher\'],passed=passed,max_abs_difference=float(np.max(abs(p-q))),class_disagreements=int(np.sum((p>=.5)!=(q>=.5)))))\n    pd.DataFrame(rows).to_csv(OUT/\'validation_restore_audit.csv\',index=False)\n    del m;keras.backend.clear_session();gc.collect()\ndisplay(pd.DataFrame(rows))\nif not all(r[\'passed\'] for r in rows):raise RuntimeError(\'Restore failed; test not evaluated. Kirim audit, jangan longgarkan toleransi.\')\nRESTORE_PASSED=True\n\nfrom sklearn.metrics import ConfusionMatrixDisplay\nimport matplotlib.pyplot as plt\nRESULTS=[]\nif EVALUATE_TEST:\n    if not RESTORE_PASSED:raise RuntimeError(\'Complete validation restore audit first\')\n    unchanged()\n    for run in RUNS:\n        dest=OUT/(run[\'student\']+\'__from__\'+run[\'teacher\']);dest.mkdir(exist_ok=True)\n        donepath=dest/\'EVALUATION_COMPLETE.json\'\n        if donepath.exists():\n            done=read_json(donepath)\n            if done[\'lock_sha256\']!=objsha(LOCK):raise RuntimeError(\'Test result lock mismatch\')\n            for name,h in done[\'output_sha256\'].items():\n                if sha(dest/name)!=h:raise RuntimeError(\'Test output changed\')\n            result=read_json(dest/\'test_metrics.json\')\n            table=align(pd.read_csv(dest/\'test_predictions.csv\',dtype={\'sample_id\':str,\'patient_id\':str},float_precision=\'round_trip\'),FRAMES[\'test\'])\n            for k,v in metrics(table.label,table.probability).items():\n                if not np.isclose(v,result[k],atol=1e-12):raise RuntimeError(\'Saved metrics mismatch\')\n        else:\n            print(\'TEST\',DATASET,run[\'student\'],\'<-\',run[\'teacher\'],flush=True)\n            m=restore(run);p=predict(m,run,\'test\');del m;keras.backend.clear_session();gc.collect()\n            result=dict(metrics(FRAMES[\'test\'].label,p),model=run[\'student\'],teacher=run[\'teacher\'],dataset=DATASET,\n                training_seed=42,split=\'test\',method=\'supervised_control\' if run[\'teacher\']==\'none\' else \'ssid_inspired_feature_kd\',\n                lambda_feature=run[\'config\'][\'lambda_feature\'],source_run=str(run[\'folder\']))\n            table=FRAMES[\'test\'][[\'sample_id\',\'patient_id\',\'label\']].copy();table[\'probability\']=p.astype(float);table[\'prediction\']=(p>=.5).astype(int)\n            table.to_csv(dest/\'test_predictions.csv\',index=False);save(dest/\'test_metrics.json\',result)\n            names=[\'BI-RADS 1/2\',\'BI-RADS 5/6\'] if DATASET==\'inbreast\' else [\'Benign\',\'Malignant\']\n            fig,ax=plt.subplots(figsize=(5.6,5))\n            ConfusionMatrixDisplay(np.array([[result[\'tn\'],result[\'fp\']],[result[\'fn\'],result[\'tp\']]]),display_labels=names).plot(ax=ax,cmap=\'Blues\',values_format=\'d\',im_kw={\'vmin\':0,\'vmax\':int(FRAMES[\'test\'].label.value_counts().max())})\n            ax.set_title(f"{run[\'student\']} <- {run[\'teacher\']}\\n{DATASET} test; seed42; threshold0.5",fontsize=9)\n            fig.tight_layout();fig.savefig(dest/\'test_confusion_matrix.png\',dpi=180);plt.close(fig)\n            names=[\'test_predictions.csv\',\'test_metrics.json\',\'test_confusion_matrix.png\']\n            save(donepath,dict(lock_sha256=objsha(LOCK),output_sha256={n:sha(dest/n) for n in names}))\n        RESULTS.append(result)\n        pd.DataFrame(RESULTS).to_csv(OUT/\'test_summary.csv\',index=False)\n    df=pd.DataFrame(RESULTS);display(df[[\'model\',\'teacher\',\'precision\',\'recall\',\'f1\',\'macro_f1\',\'specificity\',\'accuracy\',\'roc_auc\']].round(4))\n    deltas=[]\n    for r in RESULTS:\n        if r[\'teacher\']==\'none\':continue\n        control=df[(df.model==r[\'model\'])&(df.teacher==\'none\')].iloc[0]\n        deltas.append(dict(model=r[\'model\'],teacher=r[\'teacher\'],dataset=DATASET,**{\'delta_\'+k:r[k]-float(control[k]) for k in [\'precision\',\'recall\',\'f1\',\'macro_f1\',\'specificity\',\'accuracy\',\'balanced_accuracy\',\'roc_auc\',\'average_precision\']}))\n    pd.DataFrame(deltas).to_csv(OUT/\'test_kd_vs_control.csv\',index=False);display(pd.DataFrame(deltas).round(4))\n    unchanged();print(\'Selesai enam run test. Tidak ada training ulang.\')\nelse:print(\'Audit-only: test belum dievaluasi.\')\n\nREPORT=OUT/f\'B06C_{DATASET}_seed42_test_report.zip\'\nsave(OUT/\'report_status.json\',dict(completed_test_runs=len(list(OUT.glob(\'*/EVALUATION_COMPLETE.json\'))),expected_runs=6,\n    restore_passed=bool(globals().get(\'RESTORE_PASSED\',False)),test_enabled=EVALUATE_TEST))\n(OUT/\'b06c_evaluator.py\').write_text(EVALUATOR_SOURCE)\nwith zipfile.ZipFile(REPORT,\'w\',zipfile.ZIP_DEFLATED) as z:\n    for p in OUT.rglob(\'*\'):\n        if p.is_file() and p.suffix in [\'.json\',\'.csv\',\'.png\',\'.py\']:z.write(p,str(p.relative_to(OUT)))\nprint(REPORT)\nif DOWNLOAD_REPORT:files.download(str(REPORT))'
EVALUATOR_SHA256='976bd999d79e425fb5d780645934854917a998a621502de7d6177ebf03603959'
DATASET='cbis_ddsm' # cbis_ddsm | cmmd | inbreast | mias
EVALUATE_TEST=True
DOWNLOAD_REPORT=True
RESEARCH=Path('/content/drive/MyDrive/Research')
SOURCE=RESEARCH/'outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1'/DATASET
OUT=RESEARCH/'outputs/benchmark_4datasets/b06c_feature_kd_test_v1'/DATASET
REGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
EXPECTED={'cbis_ddsm': {'inference_helpers.py': 'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876', 'protocol.json': '0de8c1453a64557e34eeed639b9a719eadd4ba6bd6fc491189d6d8b39a099efd', 'b06b_engine.py': '1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487', 'nasnet_mobile__from__densenet121/seed_42/calibration.json': 'aeab55fa4d84cee601c6a8260cdf6c1cf5fdce04209a1628d06d41bc38bbe36a', 'nasnet_mobile__from__densenet121/seed_42/config.json': '6d9d750cb753f9d23b63d6904b99bccc5bbc258d2707b3213c3759a468bb181c', 'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv': '2c043d66f6bad232268594d6c2f7b9fcc20f6e85a6bd7dca01753a0bead689d3', 'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json': 'c1f99df526d8c09a8e25f19e6131e4186a4bc6407ab5fdca532cef1e3d282909', 'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json': '9c753b24d4b67dc8e5e1e1d5c928a1b03732e680b3516bcb3a394c4eb7bcbd01', 'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv': '4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76', 'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json': '355aaaf73514b5bd248f825608cf9b787ba390fc1a2617c81ce8eaff870417d0', 'nasnet_mobile__from__densenet121/seed_42/head/history.csv': 'a4b55d61c0e089a229407a4c453fca8efc30a520404ceb920e3143d015fa82e7', 'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json': 'b2d31770ee71f8c3d057d2f181f8a051b577085ed341521b1ce2d80dc175d8bd', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv': '2c043d66f6bad232268594d6c2f7b9fcc20f6e85a6bd7dca01753a0bead689d3', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json': 'abae8d02f65c4575bfbecf6f83d3a3d217f0de3738382e1bb261cc149e476c56', 'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv': '130018e3b2fa624b84a1734a050e7096b9d05f3edaa982f299a7e2a869ba5caf', 'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json': '4d53338ba09fa3c570cbe2cdcfdca283d90fc4e74357ac737e67800eb1c960d0', 'mobilenet_v2__from__densenet121/seed_42/calibration.json': '01d3be5f3b2ed72268f3e8f8347a5147b8900f342f834be4e67778854a004980', 'mobilenet_v2__from__densenet121/seed_42/config.json': 'a7ae1e4a5dc4bbe525374a1c1f83b6af53158cf0b95722bc95d620be772bdd88', 'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv': '7fde55c146d296fd4c304d70e6e276f728a56e6864ba250cf75f06a029fcfc9c', 'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json': '66cdca96fe1bd5563b8d42435e2267ff3e5d9cf42dd66a6c0678af579b1046f1', 'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json': '53141539528d9d97ef26b3207453adb629c4bba710004dfd2d22c64ee21a58ff', 'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv': '7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4', 'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json': 'a4b1e50be42719160485617f3b7f061a86ae96aa2c51cccba8b0712152bdd954', 'mobilenet_v2__from__densenet121/seed_42/head/history.csv': 'b5d36e828e8e10829b297967fe711d8a4d9bb745f576189f8c3130b6d07aa6ea', 'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json': '2d3732d2a54a01153a15260353da4b008f2e4524cb132fc66c65559e35634b51', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv': '7fde55c146d296fd4c304d70e6e276f728a56e6864ba250cf75f06a029fcfc9c', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json': '1f53afd0a42ecd1b3269cfb2be70e4c109f73473d8e33a78cbe90ce0942d8c95', 'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv': '0e3736b9efa7f51ca504da43d78579ea169a7bcad0d849cc50bd9d30f93390a6', 'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json': '1c4560021a5d18c2b5baac61297bd71dde0f559f80d8f721d1ab2b652d625a0b', 'nasnet_mobile__from__vgg16/seed_42/calibration.json': '5d9a0f6675ea9462fdc50bef4757d17bd48a290c983f307da767070a67f85909', 'nasnet_mobile__from__vgg16/seed_42/config.json': '7469d9adf43d73cf136ec531f809ee38171b06dfee5740a95bdcfe745180c894', 'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv': '3c821c87753091b41f2f05ecc6f854431e3aa5e94ec7972d60365a5fb22bb141', 'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json': '1f56fc2446a0baab796fdfecaac8fcd75f4480b7d4db91f919f37622c237cc43', 'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json': 'f6362ba596085033c517102cdd79d3eb2b7f27d3042e1aaaf32328b024fdff9d', 'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv': '4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76', 'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json': '00e069adbc61e0f7a4f9bc74f417a5800544a7dd3ad58ec600f8d8054f6a84fb', 'nasnet_mobile__from__vgg16/seed_42/head/history.csv': '9e6c05c4afb4d7c4b7852453b9278059bfd9c0c67656ab7b5b6df645c030eb25', 'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json': '2ed3c258699133e461d8d2bc4657f1fc22e05beee1fe953a6d3874df574ffb62', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv': '3c821c87753091b41f2f05ecc6f854431e3aa5e94ec7972d60365a5fb22bb141', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json': 'ecbb6a65097b71a30a7d7dbebdddf4e1f29b0d35f150652194e1730d33ce2a35', 'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv': '901f8c82e38f52663261b29dee06ba1bede4f1b673fb74a755cf46e9cd176f90', 'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json': '24b86d46959fbf667a2ee8e6dfa58d9fbcad378be16eeebadace2415e4a4fad5', 'mobilenet_v2__from__vgg16/seed_42/calibration.json': 'f8b6c2bda6ed00c17be2326976d71cc78f5530073e9af68c92983c331165280f', 'mobilenet_v2__from__vgg16/seed_42/config.json': '1b7aa521cf538a1813be292d6d149f65d1c16f2a968c5ea14c05dfccba8f45e6', 'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv': 'cc43ff07dd9ae1fa95d10b29819b0e0e0d8152a5c58995842b09c342ce8899eb', 'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json': '70b78bafeb9a9288316b9003a0a91c0e5d962fbdd3769956b829de8522cee442', 'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json': 'f3570b00cf400922b9397360351d0513bde249bdde4c944784f488f355ce4655', 'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv': '7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4', 'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json': 'a0a8585b4f44983adb7336e341f4e4bde3e59726957b5a207311b548a6de7044', 'mobilenet_v2__from__vgg16/seed_42/head/history.csv': '3a17eecb7079486b41b404b6ec5716eac3933bc93c2e6438a1e9eed607c98cb9', 'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json': '298be2ad1d963948fbd89a94f306d95fe31671b5050ad56c10552f916710f75e', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv': 'cc43ff07dd9ae1fa95d10b29819b0e0e0d8152a5c58995842b09c342ce8899eb', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json': '23e71af40a45673561ac8031e0ad2cf50b13b10a8da3f0afbcd3e8c736fce2f1', 'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv': 'f897fcc44e66a6f54e244314e6999631f46e52c193d3a0fd32b4ab73e7bf9262', 'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json': '3c6f42436fbcb3cd0f122c817ae3a412f158ee744567bf91d21ccb0401d33b00', 'nasnet_mobile__from__none/seed_42/config.json': '01753709d0c91cd76530a21d989576c408ce30c8156d1ea278ceb4ce123fdfe9', 'nasnet_mobile__from__none/seed_42/validation_predictions.csv': 'b5d53fd897eed4c4cc4a6fdd146e8bfafeb3f8aa5e9415cca4c165bc7a9c6122', 'nasnet_mobile__from__none/seed_42/validation_metrics.json': '40130dbb93bf9bc09d93160de91b424b21a461e1a28a0d9df0908d1f94ae24b3', 'nasnet_mobile__from__none/seed_42/COMPLETE.json': 'd95d943f72162f0d90e09274da4c28d566fc227cbae5fe79cc2cc780aca099a3', 'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv': '4666881bbbfb8afc77442ea50917ce71ecac98fe31a9671498364ca6c5178b76', 'nasnet_mobile__from__none/seed_42/head/validation_metrics.json': '3e91b78bd3cc953cf12d3d4bf2e026a52756fd6d1af3f9cea2619f37450564ed', 'nasnet_mobile__from__none/seed_42/head/history.csv': '71d5add792d309e018b2fdef6615d3bdbe04fb027de5b29142439d3a5d196526', 'nasnet_mobile__from__none/seed_42/head/COMPLETE.json': '0239d1e468ef609c4cab378cc209516e2ef0b104501143a8741bffa94320cfee', 'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv': 'b5d53fd897eed4c4cc4a6fdd146e8bfafeb3f8aa5e9415cca4c165bc7a9c6122', 'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json': '61382a7c7b1b036645497f6616196ea9f376301963846df39c7a6eeaf687a104', 'nasnet_mobile__from__none/seed_42/finetune/history.csv': 'ecdd8942a32d8c2b0d0e8b4cb6f86fea5d6d5594f5e5f37564dc545ff7c13aad', 'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json': 'aee3df97befd97f396472e136aa388fd9499b088782f4ee918d79e12ec9212c0', 'mobilenet_v2__from__none/seed_42/config.json': '846e6f14a3971383a95b60f8ffd0665a09c394914d4c43ba2043b359e718853f', 'mobilenet_v2__from__none/seed_42/validation_predictions.csv': '06b422a186fc2bdff1d4582709a120a56fb202607296843d3b3974334474714e', 'mobilenet_v2__from__none/seed_42/validation_metrics.json': '7c46b8c86e2643b098660141a5642926aa4ecc5ac09e97c89e4f0b0a07e34cab', 'mobilenet_v2__from__none/seed_42/COMPLETE.json': '8a8e2438becc486a8247b87c61c5ae471db26a556808c139dd6a04e5aa1b17f4', 'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv': '7d2f2a7218482587fc806ca5a4cb88fff25cf36efe391922d8e6e7db415ddba4', 'mobilenet_v2__from__none/seed_42/head/validation_metrics.json': '0f8be37158987146dd8b3bb9a37161960f202444f99f7e2c8892e20a938541a7', 'mobilenet_v2__from__none/seed_42/head/history.csv': '2f3cc03a4f94d79a1fc4c91776139196bd55b893a89286c5ad50eff66a34ff57', 'mobilenet_v2__from__none/seed_42/head/COMPLETE.json': 'f17ffe077819f27989c3cbd1e7ea42387c8cea282f7f7283312d0cde2bc9bc4c', 'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv': '06b422a186fc2bdff1d4582709a120a56fb202607296843d3b3974334474714e', 'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json': '398ef4325a9d5c85e9c6ff01f93939139b0ac51b126e4517dde1e8ba47c126ab', 'mobilenet_v2__from__none/seed_42/finetune/history.csv': '2554aaa76d389220e3de9fa9dfb9520c1e2457b547d6b9353bbc71ae0f84f161', 'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json': 'bf3964f6c86e9fdead097b18bf09618df9d6a2db951b1407e579be847adff9ac', 'teacher_cache/vgg16/COMPLETE.json': '2cce2a54cafd73b896da5c7e272b7e8461fe92545dbb92c7e8c45420a4c2b653', 'teacher_cache/densenet121/COMPLETE.json': 'e91bde95575bc5f32a422f48c08b350b33fb41e4d031e4171fc9ddbf2cbf7b8b'}, 'mias': {'inference_helpers.py': 'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876', 'protocol.json': 'e2ed77f3a0473bf8709c70551c051b2b78ac7d16baf31736aee93e19cc412fcc', 'b06b_engine.py': '1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487', 'nasnet_mobile__from__densenet121/seed_42/calibration.json': '170729db3ce33489ad1dc085a9094a7f5758c6df79856da006e219f7969c47c2', 'nasnet_mobile__from__densenet121/seed_42/config.json': 'b375d83a08971b0050c45904d8da82bb42e26d07574675dc3dbdc497b5fe49cd', 'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv': '606df8eac9c784a9002e668326d88f82c4eae9fe9a80a9a344175112d0540d1d', 'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json': 'f1ff8905957c98a87a28e4e671629606bd8847ed73fb78e36f7a9e04ae41c833', 'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json': '2335864f0f82ab5da5bbe15a2136fa66174c1e15c155fa01baa212dfbfc0a9ab', 'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv': 'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113', 'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json': '2c944fedf494d418858f0bca10c7694b554d043c4a86456cd65c5d937e1e118a', 'nasnet_mobile__from__densenet121/seed_42/head/history.csv': 'd77aed584e79f188e0e38c6d94bd94a10184cf1c96f06eb6d4f2e617d4efdc96', 'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json': 'e75d551c26fc19c2669334dd07f037b4b1631c31a03dfddfd51b5fcd27d56e73', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv': '606df8eac9c784a9002e668326d88f82c4eae9fe9a80a9a344175112d0540d1d', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json': 'f84260ff709b5409b327595a0da8d81a1216e5ae03b2be7582586f91ff5105d3', 'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv': 'dcc1942e01f411a286385f5ecb5ddd2189beb9ee949bdde375c5d6b411b03b89', 'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json': '94f9b7a87ae3b6eb4ec12ce26b0c51ea6d4391bab144debfc557dc881a0f9d71', 'mobilenet_v2__from__densenet121/seed_42/calibration.json': 'e24271150a954dca895f0d3898189c1c8f88d37df8256cf6d19730ada71145da', 'mobilenet_v2__from__densenet121/seed_42/config.json': '3dba5e2d3d551ae65d1895cb79103a6c3f8447fe2c280a716b2844f1b08cc9bd', 'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv': '6fb5fe028980b038243e5806c17e3f525ea2cd2ea00cdf4799f222b3352e6511', 'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json': '9fa759f7f0cacffc0bf780467b137e7319012858b584bc80214e857aeacac252', 'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json': '94de9b54dcb5fd7996419e131c9b339d96a56cc93772caaac6bbcf3f1b4b4856', 'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv': '558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044', 'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json': '21d6213ea31062382940efb9105614043d134416344ec5d34cc0c1ace358e9a9', 'mobilenet_v2__from__densenet121/seed_42/head/history.csv': 'be71246b8eeddf1472a995fe7523f2c8c1cfb9df0920c543834dba9b7527834c', 'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json': 'd54a7cf29e53624b4a45323646aa48c37d0611032ea4e668ef20322e8ac0604c', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv': '6fb5fe028980b038243e5806c17e3f525ea2cd2ea00cdf4799f222b3352e6511', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json': 'd58f315da48b7a1de46af421d83de3301398efc9d4af731b662ce3a2eebb9218', 'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv': 'fca2338866a83a97961d14b20ff46fae800eb72aecece79b5a2bc87f3070dd38', 'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json': '39ba5c43b523248f1e558d00145faf9892fa1dc231ab7487c909a80a1351f490', 'nasnet_mobile__from__vgg16/seed_42/calibration.json': '18c39f057a2810a9c35e1333b55d1adcdda929d0ee795f08949df7e3e24e68b1', 'nasnet_mobile__from__vgg16/seed_42/config.json': 'b58688b7826ffe20ed077a8e16f726bf91b7fd7c4e4a5991c108de6692935c76', 'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv': 'cfe3159308c2bbf6aa1ea972444f79d399d875f6e753eb4390d41236098e48df', 'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json': '5226deca661e354fd1f39b135ea617aba2ddbb3a385fc20a99777f0b3c024b8f', 'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json': '555a10c544641d98f9fb9df011dfd6e19f4d868fc765ae9f8bc7a094173e4e14', 'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv': 'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113', 'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json': '938bfb4a4c4b4520ce8fc44af2cb5a7ba23e5f983a10dded1453a3656b321e2e', 'nasnet_mobile__from__vgg16/seed_42/head/history.csv': '600995669ef918995eacd23cb553950a2cac07720e9bb3abb54994ed5ec31f97', 'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json': '030fbcf3ed99c684e6fa9ef91d8c17cedf2017f8fca7e52c451698e428f938aa', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv': 'cfe3159308c2bbf6aa1ea972444f79d399d875f6e753eb4390d41236098e48df', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json': 'bfb4296c20fa98ee895840d46b1f967f0644d3e7290174776a6264ec5fa11b88', 'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv': '22712690576890a5d2552cf60102b058e848f6ea18eda69f18689c95189292e3', 'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json': '8f9567f277b8ee1f11f1722e5d60fe29c98bef36a6394be161c5c0dd981dd125', 'mobilenet_v2__from__vgg16/seed_42/calibration.json': 'e431cff27db5235effcccc877b509cae79fe6b3dc44dc0b3d59af077f3f2c996', 'mobilenet_v2__from__vgg16/seed_42/config.json': '32dca61b64b657870ea470392423a92cdebd0c3535c6d67e3d47faa827dafbab', 'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv': '485fb2170397bd148110c9e4c856f19e2319dbb039800d1baf74a90536e6dd12', 'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json': 'af7d25e755232ba9510bd18e45d120b69e79cc2e41ccea64f3700f9e1ac05cb9', 'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json': 'df6c10579454987bbf1bc61ee5348647d01faf762297f9719b0bb7cfd3925996', 'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv': '558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044', 'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json': 'b34c07a6ee650482e60313227e9572ba1d393212029e7eee30aebf3dc0f4e6df', 'mobilenet_v2__from__vgg16/seed_42/head/history.csv': '70ab34a10362b29c39bcfdf72b6ea19fe90a333590341a8df50bbf5dc713e7fb', 'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json': 'a5465c4e888c77fae824b4e5c3b4ea877f4cbb931e42dde525660f5eca9ac007', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv': '485fb2170397bd148110c9e4c856f19e2319dbb039800d1baf74a90536e6dd12', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json': '4252a862aba8186176c310392747b7ab7e808c78bfc0df6c8e0280119d2086d1', 'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv': '1d4cbadaf1797bdb21c1f2ec289c4d37c48423387e16100027e5f72faab8b0ed', 'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json': '02b9ef78ce26b0cdb0ec37de1597a25c1463f9c8fa0fbd5d0d3804bc458c919b', 'nasnet_mobile__from__none/seed_42/config.json': '3cff9a8084913b5abf8fae84b19f11b8009d2c124deda0a5dc2e0e5ffeecb8e9', 'nasnet_mobile__from__none/seed_42/validation_predictions.csv': '5bfdcf278205b7222a6f534974fc4e402e36b09e4f4631f5d1a5b55a5fe34263', 'nasnet_mobile__from__none/seed_42/validation_metrics.json': '0f48694d80b21c3c74a3971b68395e3e264ac7f264df6463fb3ecb83074dc964', 'nasnet_mobile__from__none/seed_42/COMPLETE.json': '48802486ae91b6336cb8e835c871259deaa7d3819f983cd7a62d68f364298bb1', 'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv': 'f66c872b01f5e4c614583194db8510f8f85133bb135f1b34ce98cba07cc89113', 'nasnet_mobile__from__none/seed_42/head/validation_metrics.json': '6f9f218a0ddac8d0e84c731506a9a62d0ba5905163bf3b1e4c996703297e6f98', 'nasnet_mobile__from__none/seed_42/head/history.csv': '58e0e1aac4f9e46d956a8af1be921fcb492e3e5838e4ba26b82caa4636ca9078', 'nasnet_mobile__from__none/seed_42/head/COMPLETE.json': '6f3b2f3f50e47f32dccadafb5a74afb41e6e4c52dc8aac462df3bd55cbf1e464', 'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv': '5bfdcf278205b7222a6f534974fc4e402e36b09e4f4631f5d1a5b55a5fe34263', 'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json': '368737cef28eaaeba33d11db11c62fe9652fc571cb39282631dae44a8334f4d2', 'nasnet_mobile__from__none/seed_42/finetune/history.csv': 'a5bac70b6cb4a49a2b089183b572122ebc3ec578ff1d29cf53da6b05b5bd3bf9', 'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json': '6e80b4c27584fc04e33ebea2f42eff92fd4d3fc225a63d544fe0b5e4103ea120', 'mobilenet_v2__from__none/seed_42/config.json': '0ac93d71ce607da5b43a11a05a2553c758fb19bb1e63ad5f097f6c4a5fe11fc4', 'mobilenet_v2__from__none/seed_42/validation_predictions.csv': '2447eb121c86694080a7c1d824a66350fc850d5cd35f8482ecf4e81814999538', 'mobilenet_v2__from__none/seed_42/validation_metrics.json': 'cc9b05219bfe09e39a31544c6063286842cbcf88113f0ce7bfa6698f1c375c99', 'mobilenet_v2__from__none/seed_42/COMPLETE.json': 'e6b1988b8d48ff44709227cde1c482b47660911c5d8bffa590da51aafdc932cc', 'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv': '558ba9d06be08265489e3bc5aa66851d692f5f34f65b8ada581160683c862044', 'mobilenet_v2__from__none/seed_42/head/validation_metrics.json': 'c7735a30e5d72ad75c580d926ae751737f40ad31a7f0cc63942f3ed76fb22502', 'mobilenet_v2__from__none/seed_42/head/history.csv': '22bc8564cbaa6d075e445bd893c0426925f488be63a667490a3852f3296643b3', 'mobilenet_v2__from__none/seed_42/head/COMPLETE.json': '1c49e26b935b4fdc0b73fae04d8158740dab5fffe7e43302c5e0873d318574b3', 'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv': '2447eb121c86694080a7c1d824a66350fc850d5cd35f8482ecf4e81814999538', 'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json': '4d918e7c1af0b8cb3ce7c899e116060b20c596a409119144d87804fd978e9196', 'mobilenet_v2__from__none/seed_42/finetune/history.csv': '0a3001ffab045aada980d973cdb94a746ef291f547820e51d374ab4b51bd452f', 'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json': '0bec00534ac3cc6a7bdebef15b729c1084d1ca297b36284664658c412f602e6e', 'teacher_cache/vgg16/COMPLETE.json': 'd7e9cb02aa9c03918d665234b7ad77dc12964907df7b1133b96517a7b1d449fb', 'teacher_cache/densenet121/COMPLETE.json': '1c6e3afc06faa08b88b62825e84992f89c29ce8570b2033cfa38e84f2794242c'}, 'cmmd': {'inference_helpers.py': 'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876', 'protocol.json': '994a7c6118652885a1e42d6c09181f24bed8be08ed90a4b3258dccc0f5321d77', 'b06b_engine.py': '1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487', 'nasnet_mobile__from__densenet121/seed_42/calibration.json': '40ee044aef90bd7b86c7578249d45a518d32f32704fe475ab934532a1765377b', 'nasnet_mobile__from__densenet121/seed_42/config.json': '015fab1a4226c90d7ec799263ec9e30daa9d20fbaefca2edf1c3381d30d7e808', 'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv': '7c6edf398625de8fd0d027b2b13c223190817e3ef54ee3092f466b87b6e92e86', 'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json': '951051d9634df0da67e50dba50023ca134ea78f2df1b9608d2917137500ed184', 'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json': 'c69d02685b3c342da6c7ab699f12f1b8ffd51befe6918c8cb3ce8ceff5dcab1b', 'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv': '12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df', 'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json': '8897110e2edbddd3ff1b7a55422b8daf77a7dcb622c4301cc1907c55fd9f4b1b', 'nasnet_mobile__from__densenet121/seed_42/head/history.csv': 'ae35e41635088d64e00513fb304f4fc71438c055a25eedd2e3fdc1f3e35df6f6', 'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json': '89e288222f79e468797e9ce1264458e0f6570009eb7db80751f54798abf6d6e3', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv': '7c6edf398625de8fd0d027b2b13c223190817e3ef54ee3092f466b87b6e92e86', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json': '12700491247aa7e7933f1b8b311afb9cff4f80e89d3f5637e5d520b24931151d', 'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv': '2b45c733ecf4a2c6830b4dd0b72e17754a58cfdb5df6e08c6092a6f940c6f407', 'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json': '01e7b8e24719d73db2d2c5df535fbfbc039f5bd52d2df126eee91f1fba7359e9', 'mobilenet_v2__from__densenet121/seed_42/calibration.json': '9b397ca23b3ddbafa86d981ebc53f1a35b2a2c8ef3ba2cf9f1045fbcaf7a235d', 'mobilenet_v2__from__densenet121/seed_42/config.json': '84d5ea3ce34c1cd57fede737dcab15eca154d3e513265bdff78b3ebed7036c9a', 'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv': '975a2ea6bb724c05fc0c2713e1f6788bc80d33a256ddb3b4e06f045c80dd398c', 'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json': 'e5027aad97bff50f13d5e8e57fef7a01fa6782fb0220ab7c18df01c800351f1d', 'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json': 'c3c9db59e25b40b54fbc86b1aa614975f2279a56c840af833bf596784f3e14e8', 'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv': 'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe', 'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json': '88e1358a2e8d153752614b00bf35192b80b000e483df4d1d925e018ea0944be0', 'mobilenet_v2__from__densenet121/seed_42/head/history.csv': 'b9dd2da2c4cba2a8b40154e87b9471758eb252089876387142c9bedf3abfc53c', 'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json': '881c6aa28b1f0972d73ddc301aff889e943491dac6942c795f2e2f2ba151d5d3', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv': '975a2ea6bb724c05fc0c2713e1f6788bc80d33a256ddb3b4e06f045c80dd398c', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json': 'f2b43de1dd37de927585f9d925af9cf56a5c0808aafda52a3aad2287ea421ada', 'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv': 'fd6b8236f2d98b31b3348097e488fa7999d7f34a9b86d52439c353017b877ca2', 'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json': 'fd416cfba65a87338fdc07315d843c63ba4c8c4163f472b62d6b3c095330972e', 'nasnet_mobile__from__vgg16/seed_42/calibration.json': '7374826c8afcf59f262fd1f29901e3176baf60581b9ac80aa0c247b350954a83', 'nasnet_mobile__from__vgg16/seed_42/config.json': 'ceb47867c479168c8193c21c6914b7a04dc5f1c7beeabacd15203bb4d4ba4bad', 'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv': '6287d0782cc656314b5c7f3acbe9b71717fc25812441eaafb9390739e85ed5da', 'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json': 'cde0d42ab9bab32fac7977603320e89d23405b5b8d36cccab0c8bf4bc4e6d1a7', 'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json': 'bf22344b0319f5eb0525babfee8fc824a16b8a5c474e075462d1c31e34c7070c', 'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv': '12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df', 'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json': '810bf5d63d356dbdb85d61a47a50cb13e2805310a07349e56957667bf6610c7d', 'nasnet_mobile__from__vgg16/seed_42/head/history.csv': 'e1ee3fd61193503cd14cd7a35b94a42bcdf9a224243dbc6f55a7e829f4050fb1', 'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json': '8c4d7d83f385b2f16a1a959c3bb0377644845180b2f61b977c8b1c00f16b0189', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv': '6287d0782cc656314b5c7f3acbe9b71717fc25812441eaafb9390739e85ed5da', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json': '2dac829e61099783bbcde8a786a597c23128c2c64392eea776d35450022ed846', 'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv': '7c94380d04a184e2ffb610ce55a6ce6dee23380e1176e67186e0a4988f825da0', 'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json': '4f0fdce1dc2d5ff5ed6d1c99beaa960fdf9cd4d51ef18defa2a9a62273e8a03c', 'mobilenet_v2__from__vgg16/seed_42/calibration.json': '38a4ffbbbbcd026e91a91a7bed61235f93e607ec90c867f71f49459a44668d85', 'mobilenet_v2__from__vgg16/seed_42/config.json': '670f81e29a287b8192fe2386dcf32088c63facfd4e7cc518c02170997c6e9f68', 'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv': 'e7c7a425cd0d0704e9626c7a810349d12835b8ba2079fa17497c4fd439340e81', 'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json': '67c4cdf07a5d9148819d5d5843f39c8a66dd102c1891fca3c33a695674c231f1', 'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json': 'a65f9476970326e198b95381af507b93a1ff43470fd2e207cb0f0f11b5468694', 'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv': 'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe', 'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json': '5f0b0b34b5690e0c1a8706733cdc600061799d1ac9a5f85e446a06ad411f54a8', 'mobilenet_v2__from__vgg16/seed_42/head/history.csv': 'e42bca4ff948735df98b02a474c39a090a9e7a2de53b56f2a782cdc8f54655a9', 'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json': '0273387406c8d8ba37ca1925fd98b47ca1e2111f3039385afe3ba9d7df25fd60', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv': 'e7c7a425cd0d0704e9626c7a810349d12835b8ba2079fa17497c4fd439340e81', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json': 'ff4cc92399471f8919ac0fb6f5f576ed357266e574cf6e155596fbad63f1f3ea', 'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv': '039db4de369960bcf3d5b2ad9a716e1a065950e22723444faa2a9f0c71fe13ce', 'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json': 'ae20cd272d93e895caa67263c13bab879702b341e53194608b6f9504a03b20d6', 'nasnet_mobile__from__none/seed_42/config.json': '84fb9053e6df544e7682c3740e231d9a0d2d9f60d97f14be96ec8e90a3abf870', 'nasnet_mobile__from__none/seed_42/validation_predictions.csv': '78947db308cfbdd255341993052107d1b389a72f3c32f33c41971acbaf954ebf', 'nasnet_mobile__from__none/seed_42/validation_metrics.json': '9e85707f820b2badc0ce94a144563f91faaa7fe808357a4db3e2970cd58b707d', 'nasnet_mobile__from__none/seed_42/COMPLETE.json': 'f54d0ef9209d01def6d538a489161b785f86743f56c414b864344aaefad6a2e2', 'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv': '12f44c4796749e797823ad0be3d7a1d15ec1134f775cf0d2d9253fd16b5769df', 'nasnet_mobile__from__none/seed_42/head/validation_metrics.json': '88a87226154ab3434a45df2845b0631fe5a575646d25be3c24727d009558a752', 'nasnet_mobile__from__none/seed_42/head/history.csv': '65d31a3de51f91fbd59b2f934a8eae28787bfc296356b2e1d14e1a3db40cb35e', 'nasnet_mobile__from__none/seed_42/head/COMPLETE.json': '97798d8d14076613a76f072e1a1f21e1862e6d0e316412412baad333e398ecee', 'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv': '78947db308cfbdd255341993052107d1b389a72f3c32f33c41971acbaf954ebf', 'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json': '031e59683dbfe0e2a70dd5597adb24f2db07870cfc188946b27cce397327fbf5', 'nasnet_mobile__from__none/seed_42/finetune/history.csv': '6879a5c22c41410134359315c71ee14bc43fa94e6d6946e323c9e21077a06706', 'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json': 'b5b8c56a7faac7281f1ae5e8373092156e1b2f9dff89b06a82388b1bb050c6ef', 'mobilenet_v2__from__none/seed_42/config.json': '0025a1bee6e8cd0cbfd41d09fbb03cead63eb311b52fb223654e59fd3e62adb5', 'mobilenet_v2__from__none/seed_42/validation_predictions.csv': 'a0c26ee58a67f0aa2b5d24a2a02bdcc10ecfbb15832a8c484c2e3263254e9543', 'mobilenet_v2__from__none/seed_42/validation_metrics.json': 'a6e4815c2b418a187fceedc1f1c7c2c33366a843d2890216d47892b35d281ebb', 'mobilenet_v2__from__none/seed_42/COMPLETE.json': '74d9ad8f85b27faaa5807429a4f2d4ca72c3acac8fd44abfad828ed8321255dd', 'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv': 'ea656d7107fc093f2d08fc6ba93c960a6334412819b66b9aaf3ed208d7f8acbe', 'mobilenet_v2__from__none/seed_42/head/validation_metrics.json': '42af56c0726630beda2eb37179231097c955ab81ba96f3294a5eb532272ead0f', 'mobilenet_v2__from__none/seed_42/head/history.csv': '5fcee53f341317fa8e976557906a81f74d0a130135a55951a28bc1a36a1a6d83', 'mobilenet_v2__from__none/seed_42/head/COMPLETE.json': 'e64247e6773fd8b53a6bad3ac57c0599a63ae4840f2366ab8c669618161be6ac', 'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv': 'a0c26ee58a67f0aa2b5d24a2a02bdcc10ecfbb15832a8c484c2e3263254e9543', 'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json': 'ab4e79405b3ddc4c9b13466c81c887da1707437dcd7e8be7967fc540dee0312a', 'mobilenet_v2__from__none/seed_42/finetune/history.csv': '7daa72fcb212173b8f2617c4ef768c34c5eab78e2aac0820c67fcc7948ed638f', 'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json': '06bef1a97669990c3c6906b05c7278278e764e5b3cf1ace79cd7a50a2481f532', 'teacher_cache/vgg16/COMPLETE.json': '7fadb5a4d8696c83925ca06a9b3cc4dce668bfd1254060e196b089f8ddca8fce', 'teacher_cache/densenet121/COMPLETE.json': '413897bd9db8d85d31302d702a9c5073d35076635b7aaf9f513ca1c7c36faa12'}, 'inbreast': {'inference_helpers.py': 'afd9012420c7d3c0e398702b0419386fc0e6b708b9bb5bfedc85641035c98876', 'protocol.json': 'fd9074642b74e78ec16d883a80cc687484402f2ad5464b9a2a472d0c07ba5988', 'b06b_engine.py': '1ad07f638580b5ede6b57ee065e2963830db45d4a79fd7d34baead12f61fc487', 'nasnet_mobile__from__densenet121/seed_42/calibration.json': '7c2485859a7045ec7b3b2d441d1286d0a78c0d5aee96797783b240c6e0f4970e', 'nasnet_mobile__from__densenet121/seed_42/config.json': '1a2f5ca656439b3231965b73dc7ee1448da81bbc627bb47af44612bfc2a9ca2e', 'nasnet_mobile__from__densenet121/seed_42/validation_predictions.csv': '215eab0f1f3d35b1b82121539cb72c7f7f87087a0da5722c0d809478af4bb61f', 'nasnet_mobile__from__densenet121/seed_42/validation_metrics.json': 'fe3a3f44855b16f492b73d78192de912e6da84460ced8a82928dbb7c12750b04', 'nasnet_mobile__from__densenet121/seed_42/COMPLETE.json': 'fe4a51a734853ca7aa964ef0c5a053c4998beceb8581f2988027414cc70c539a', 'nasnet_mobile__from__densenet121/seed_42/head/validation_predictions.csv': '1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29', 'nasnet_mobile__from__densenet121/seed_42/head/validation_metrics.json': 'bd76761a49fad9cd03a09ba793199cbc9f68da0046b78e6b3528c7fa2ebb010e', 'nasnet_mobile__from__densenet121/seed_42/head/history.csv': 'f9847c23987517b4720285dd557d223cbf5d0d206e55c3e8b5a2ae5629727b2c', 'nasnet_mobile__from__densenet121/seed_42/head/COMPLETE.json': '938ac84e64725003dd648ce334d547c063bcd6b64dbdd8a04ceff91bb83be605', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_predictions.csv': '215eab0f1f3d35b1b82121539cb72c7f7f87087a0da5722c0d809478af4bb61f', 'nasnet_mobile__from__densenet121/seed_42/finetune/validation_metrics.json': '5d3c99f2f092b1141e6efc6a6a7457b94ebb4771ee8451a1afe30c7afea2d9fe', 'nasnet_mobile__from__densenet121/seed_42/finetune/history.csv': 'db8cd48d98ff18cdfc9ec74396cb80ee18e97d365f2e852ef1e412f19e21c337', 'nasnet_mobile__from__densenet121/seed_42/finetune/COMPLETE.json': '1dfe2f0eecbf4123f9ce5b962f3286fbaea7b509e5a724e4369fd744e808ac7c', 'mobilenet_v2__from__densenet121/seed_42/calibration.json': '70633ab6b36020e024f53ce362fccfd4f44c9c5abc1be2a6bfd65bcaaba7fce4', 'mobilenet_v2__from__densenet121/seed_42/config.json': 'eb1da87bbcf14145a9b9714a39e0cfa1c5054da06a438ebfe431add8ea35142a', 'mobilenet_v2__from__densenet121/seed_42/validation_predictions.csv': 'ea3bc1497a53528a25fd205ef4d98b7bbb3b9610716f5f7877571b960ec3b581', 'mobilenet_v2__from__densenet121/seed_42/validation_metrics.json': 'e929c2d60a36f0390e7ac8e455d01e3df125fd776b92eb0505048ce4cd6ae75b', 'mobilenet_v2__from__densenet121/seed_42/COMPLETE.json': '0fe44cd5840c1e62ae8c3283f002814dc6ea90c9613042ca54d9b42adadd783f', 'mobilenet_v2__from__densenet121/seed_42/head/validation_predictions.csv': '761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197', 'mobilenet_v2__from__densenet121/seed_42/head/validation_metrics.json': '54af2389f78cda620aef1f19e8bf2b51ba8d55620b1944776bf5dfabc15168f1', 'mobilenet_v2__from__densenet121/seed_42/head/history.csv': '3ce49778bf38cfd4f8142d101b53956b8acadaca6ebfe2149b8a1841c4749865', 'mobilenet_v2__from__densenet121/seed_42/head/COMPLETE.json': '7492a3da9b994a237676240b90e39b183843e3fb76b7fd05aa1f19113556674b', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_predictions.csv': 'ea3bc1497a53528a25fd205ef4d98b7bbb3b9610716f5f7877571b960ec3b581', 'mobilenet_v2__from__densenet121/seed_42/finetune/validation_metrics.json': '20fcc37db1fe1f703cca24ce5b4ce1106490ece0d8734343091be2bf1d066408', 'mobilenet_v2__from__densenet121/seed_42/finetune/history.csv': 'c623dac1f026bd2f1a706e6dbf132e9ff44a3c2c138eb368b7e101a3b53de980', 'mobilenet_v2__from__densenet121/seed_42/finetune/COMPLETE.json': '963175d0c6e0c8d784031657118bfe8a150adce56d3334783fcd380d9149f51f', 'nasnet_mobile__from__vgg16/seed_42/calibration.json': 'e8901cedda5acd7f222515239a221a250c5029959bfdd70d8a85737368b84258', 'nasnet_mobile__from__vgg16/seed_42/config.json': 'fa4d39634389a195a977108b4266bc00c1a5850deb7b3e470c383f3ddf71c121', 'nasnet_mobile__from__vgg16/seed_42/validation_predictions.csv': '594141687dcaa99c1c016165d0938e32843980c0caafbcf8d0222779f6f0732d', 'nasnet_mobile__from__vgg16/seed_42/validation_metrics.json': 'b0452db65a61f64c42cf9e105b42920cfe89ef6842d4627260bf3010e530a1a9', 'nasnet_mobile__from__vgg16/seed_42/COMPLETE.json': 'f32ac38b692da60dbfebde3763da432205aee39f38e08b3033b0e8de411e36aa', 'nasnet_mobile__from__vgg16/seed_42/head/validation_predictions.csv': '1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29', 'nasnet_mobile__from__vgg16/seed_42/head/validation_metrics.json': 'cd1e45def4f364adfe4c2bd5195f27a1cd8a404d7dccee857975ddfb50274310', 'nasnet_mobile__from__vgg16/seed_42/head/history.csv': 'da3e2d90a8e16e895e6ac7d03fbdd510d5d09b3c40b44f0e60c0cbf7b685d409', 'nasnet_mobile__from__vgg16/seed_42/head/COMPLETE.json': 'b2df91949e02c87fb45b5a0c377ed2f1969c73a305e57650b1d77b8be4d01791', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_predictions.csv': '594141687dcaa99c1c016165d0938e32843980c0caafbcf8d0222779f6f0732d', 'nasnet_mobile__from__vgg16/seed_42/finetune/validation_metrics.json': 'cd5d225d04e937abf29b2ef4a29a18c0a9db0ee52f6766b5e1afc320fcd36d72', 'nasnet_mobile__from__vgg16/seed_42/finetune/history.csv': 'ebb3c81b8cc089806765624a541d878dfcf05b023d6a0667192053cc77d780f4', 'nasnet_mobile__from__vgg16/seed_42/finetune/COMPLETE.json': 'feb2e8a4a57b74c859a6484f1be316a46ea17b869504ab12a02cf614b1713daf', 'mobilenet_v2__from__vgg16/seed_42/calibration.json': '18235f0d15caa10ed6fba88eaffc771b2688960eaf24cffea9ed0512b5ebb2fb', 'mobilenet_v2__from__vgg16/seed_42/config.json': '7fd5f8e90cd65f0e5ab5f01fe909729a65cc8190da905e0fab4da4b879eec7c0', 'mobilenet_v2__from__vgg16/seed_42/validation_predictions.csv': '4fd490f53970e20659a95841ceb44168842e305740b612a481dfb6c2f4829e67', 'mobilenet_v2__from__vgg16/seed_42/validation_metrics.json': '0ec2fd2a1c6a03ab46825ba0a97e7d40b8f6688faced966eb01ecae724187527', 'mobilenet_v2__from__vgg16/seed_42/COMPLETE.json': '75c248e2a342949fe45b0fa446db3d4ed317397f62d4ed4301d9a49ce108a3cd', 'mobilenet_v2__from__vgg16/seed_42/head/validation_predictions.csv': '761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197', 'mobilenet_v2__from__vgg16/seed_42/head/validation_metrics.json': '731e9cbd09caf95d0a57f455011aad41d160ad09f88f0091b0094c1405257f0a', 'mobilenet_v2__from__vgg16/seed_42/head/history.csv': '0863711ced936eeae3ffede6b4f0370de2d4d51436dae51273f95d93d84632a3', 'mobilenet_v2__from__vgg16/seed_42/head/COMPLETE.json': 'b4ad183401803738f2140c5e974cacfc1deace3c08499138f649ddde9978fae2', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_predictions.csv': '4fd490f53970e20659a95841ceb44168842e305740b612a481dfb6c2f4829e67', 'mobilenet_v2__from__vgg16/seed_42/finetune/validation_metrics.json': '0c140234e242032d2b09f4dcfb9f56dfd285d3170416b428a0b670cd3a80b041', 'mobilenet_v2__from__vgg16/seed_42/finetune/history.csv': 'ebec73e6fe896c9c4abeed46dc9d7f6719cd14c1804675b051061141edfd1412', 'mobilenet_v2__from__vgg16/seed_42/finetune/COMPLETE.json': '7e67c01093207c446709026819208605646b76cd6a23087eba3605b42b0f110a', 'nasnet_mobile__from__none/seed_42/config.json': '30d1a8d844f5987fc1bf12edc0a12891928608e45627b8092eea7dce3897fc07', 'nasnet_mobile__from__none/seed_42/validation_predictions.csv': '4c7df2289f8da997b65ad874aec78e25eb4cc9111c5022b69800061b9ad4f1a2', 'nasnet_mobile__from__none/seed_42/validation_metrics.json': 'cd4064bf792618234aa243ca4efc830d577cacade26a30fd671ff2cb51114179', 'nasnet_mobile__from__none/seed_42/COMPLETE.json': 'e04056fbc448366c9daf5081742bf4e8f725c4a59f78dff4c936fac45267e98c', 'nasnet_mobile__from__none/seed_42/head/validation_predictions.csv': '1c7a79e140029ed7b2c3042f7e9b1c04e37e2f0b87ecb0a2335df01032df6a29', 'nasnet_mobile__from__none/seed_42/head/validation_metrics.json': '54fa1f99208e7a2acaafff7dc256778e6fee03c323019f4869654a1d9c7db0ee', 'nasnet_mobile__from__none/seed_42/head/history.csv': 'ed655cca3e4164110acdbbf51f44cbeee86c1cdaced39c8d8358d9f778b00fa3', 'nasnet_mobile__from__none/seed_42/head/COMPLETE.json': '06ea3c4da1501d01e96aaaf999acee615684c7957284480c18ba8c707f53b475', 'nasnet_mobile__from__none/seed_42/finetune/validation_predictions.csv': '4c7df2289f8da997b65ad874aec78e25eb4cc9111c5022b69800061b9ad4f1a2', 'nasnet_mobile__from__none/seed_42/finetune/validation_metrics.json': 'ea1e85afd3f288a4ed8479dbf82f41abaeab1050ad1642dd5cb38b7972d9403c', 'nasnet_mobile__from__none/seed_42/finetune/history.csv': '4e46a32c16b9ea30b4405160289ef06654c9c054e5fdb5e2ae9333ccac966896', 'nasnet_mobile__from__none/seed_42/finetune/COMPLETE.json': '3e40d7aad9cf614d7d4250d85724c5c5ee9e003e82d6389ca5e42df1d16196ea', 'mobilenet_v2__from__none/seed_42/config.json': '480b25a6406d8037fe5a0e74785c70d56d1cce89d7109ddd19d79a97ffa8743c', 'mobilenet_v2__from__none/seed_42/validation_predictions.csv': '41054a104fee39c7609029b470c9dc805413b80593b8d102fedf7e4188f87895', 'mobilenet_v2__from__none/seed_42/validation_metrics.json': '68f4bf60db27ba0dc8e48cf83a9d8de7c62d718a8c096c741242ea9c7859f573', 'mobilenet_v2__from__none/seed_42/COMPLETE.json': '858f64b58128a4f54f461b55548f2cc13068a504d02a3ebcf3657fd659179a9e', 'mobilenet_v2__from__none/seed_42/head/validation_predictions.csv': '761ef4696f78f9146d1e0271ce88568eb0219cc05747ffd54a5f3117cb71b197', 'mobilenet_v2__from__none/seed_42/head/validation_metrics.json': '49178d1f51a961fa64153d8e9e4ae1f6891d7744ff2844f5c77e5c14deb2e4cb', 'mobilenet_v2__from__none/seed_42/head/history.csv': '411b8cf1be9cc4104313d75149eb957b54199f593f686ec25f196dc92493fc2e', 'mobilenet_v2__from__none/seed_42/head/COMPLETE.json': 'e1ca734d4b1b65a6d3150a8273e654195c5ef570c35ff350774b8895b81ba8d0', 'mobilenet_v2__from__none/seed_42/finetune/validation_predictions.csv': '41054a104fee39c7609029b470c9dc805413b80593b8d102fedf7e4188f87895', 'mobilenet_v2__from__none/seed_42/finetune/validation_metrics.json': '1f2ec3ed235830a847529554351695c30563aa277eb8549a4d95d41b44b5f975', 'mobilenet_v2__from__none/seed_42/finetune/history.csv': 'a89a2490311368d27771fa477706912bda982b01af8b876e47ec6e7f62e59c17', 'mobilenet_v2__from__none/seed_42/finetune/COMPLETE.json': 'bccec9143a64e81c40137387663ed2915efe4ed0482aa602d676b3f82ec30282', 'teacher_cache/vgg16/COMPLETE.json': 'e42f62737f69a7560e1e9f2dd6a200064b4b04f02c5f945bfd050cd889ada97e', 'teacher_cache/densenet121/COMPLETE.json': '428da9889dfa70c5feabc524e32da88f9e0ce552c0a7a8a53429399002ae2980'}}
if DATASET not in EXPECTED:raise ValueError('Unknown dataset')
if SOURCE.resolve()==OUT.resolve() or SOURCE.resolve() in OUT.resolve().parents:raise RuntimeError('Output overlaps training')
def read_json(p):return json.loads(Path(p).read_text())
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
def objsha(o):return hashlib.sha256(json.dumps(o,sort_keys=True).encode()).hexdigest()
def save(p,o):
    p=Path(p);tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(o,indent=2,allow_nan=False));os.replace(tmp,p)
def guard(p,o):
    if Path(p).exists() and read_json(p)!=o:raise RuntimeError('Existing lock differs: '+str(p))
    save(p,o)
SOURCE_HASHES={}
for name,digest in EXPECTED[DATASET].items():
    p=SOURCE/name
    if sha(p)!=digest:raise RuntimeError('B06B differs from reviewed report: '+name)
    SOURCE_HASHES[str(p)]=digest
PROTOCOL=read_json(SOURCE/'protocol.json')
helper=(SOURCE/'inference_helpers.py').read_text()
helper_path='/content/b06c_inference_helpers.py'
Path(helper_path).write_text(helper)
linecache.cache[helper_path]=(len(helper),None,helper.splitlines(True),helper_path)
exec(compile(helper,helper_path,'exec'),globals())
# Helper only defines models, data transforms and metrics; no training entry point.
RUNS=[]
for teacher in ['none','vgg16','densenet121']:
    for student in ['mobilenet_v2','nasnet_mobile']:
        folder=SOURCE/(student+'__from__'+teacher)/'seed_42'
        cfg=read_json(folder/'config.json');done=read_json(folder/'COMPLETE.json')
        if done['config_sha256']!=sha(folder/'config.json') or done['epochs_completed']!=30:raise RuntimeError('Incomplete/config mismatch')
        if cfg['student']!=student or cfg['teacher']!=teacher:raise RuntimeError('Pair mismatch')
        for k,v in PROTOCOL.items():
            if cfg.get(k)!=v:raise RuntimeError('Run protocol mismatch: '+k)
        prefix=Path(done['best_checkpoint'])
        if folder.resolve() not in prefix.resolve().parents:raise RuntimeError('Checkpoint outside run')
        parts=sorted(prefix.parent.glob(prefix.name+'.*'))
        if not Path(str(prefix)+'.index').exists() or not any('.data-' in p.name for p in parts):raise FileNotFoundError('Missing checkpoint '+str(prefix))
        for p in parts:SOURCE_HASHES[str(p)]=sha(p)
        RUNS.append(dict(student=student,teacher=teacher,folder=folder,checkpoint=str(prefix),config=cfg))
print('Six completed runs found. No training invoked.')

## Cell 3 — Verifikasi split/citra dan bekukan evaluasi

In [ ]:
entry=read_json(REGISTRY_PATH)[DATASET]['full_512']
if entry['status']!='READY':raise RuntimeError('Cache not ready')
folder=Path(entry['path']);meta=read_json(folder/'READY.json')
if meta['dataset']!=DATASET or meta['image_size']!=512 or meta['file_sha256']!=PROTOCOL['data_sha256']:raise RuntimeError('Data differs from B06B')
FRAMES={}
for split in ['train','validation','test']:
    path=folder/(split+'.csv')
    if sha(path)!=PROTOCOL['data_sha256'][path.name]:raise RuntimeError('CSV changed')
    f=pd.read_csv(path,dtype={'sample_id':str,'patient_id':str})
    if f[['sample_id','patient_id','label']].isna().any().any() or f.sample_id.duplicated().any():raise RuntimeError('Invalid identities')
    if set(f.label)!={0,1} or set(f.split)!={split}:raise RuntimeError('Invalid labels/split')
    FRAMES[split]=f
allf=pd.concat(FRAMES.values(),ignore_index=True)
if allf.sample_id.duplicated().any() or (allf.groupby('patient_id').split.nunique()>1).any():raise RuntimeError('Split overlap')
if sha(folder/'images.zip')!=PROTOCOL['data_sha256']['images.zip']:raise RuntimeError('Archive changed')
IMAGE_DIR=Path('/content/b06c_images')/DATASET/PROTOCOL['data_sha256']['images.zip'][:16]
IMAGE_DIR.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(folder/'images.zip') as archive:
    for row in pd.concat([FRAMES['validation'],FRAMES['test']]).itertuples():
        member=row.cache_member
        if Path(member).name!=member:raise RuntimeError('Unsafe cache member')
        dest=IMAGE_DIR/member
        if not dest.exists():dest.write_bytes(archive.read(member))
        with Image.open(dest) as im:a=np.asarray(im)
        if a.shape!=(512,512) or hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:raise RuntimeError('Pixel mismatch')
LOCK=dict(version='b06c_v1',dataset=DATASET,seed=42,batch=8,threshold=.5,
    source_hashes=SOURCE_HASHES,data_sha256=PROTOCOL['data_sha256'],tensorflow=tf.__version__,tf_keras=keras.__version__,
    restore_atol=1e-5,restore_rtol=1e-5,test_status='Previously inspected in earlier experiments',
    evaluator_sha256=EVALUATOR_SHA256)
OUT.mkdir(parents=True,exist_ok=True);guard(OUT/'evaluation_lock.json',LOCK)
save(OUT/'training_protocol_reference.json',PROTOCOL)
AUDIT=pd.DataFrame([dict(split=s,images=len(f),patients=f.patient_id.nunique(),negative=int((f.label==0).sum()),positive=int((f.label==1).sum())) for s,f in FRAMES.items()])
AUDIT.to_csv(OUT/'split_audit.csv',index=False);display(AUDIT)
def unchanged():
    for p,h in SOURCE_HASHES.items():
        if sha(p)!=h:raise RuntimeError('Source changed after lock: '+p)
def align(saved,ref):
    if saved.sample_id.duplicated().any() or set(saved.sample_id)!=set(ref.sample_id):raise RuntimeError('Prediction sample IDs mismatch')
    saved=saved.set_index('sample_id').loc[ref.sample_id].reset_index()
    for c in ['sample_id','patient_id','label']:
        if not np.array_equal(saved[c].to_numpy(),ref[c].to_numpy()):raise RuntimeError('Identity/label mismatch '+c)
    return saved
print('Output:',OUT)

## Cell 4 — Audit restore seluruh model pada validation
Backbone/classifier wajib terisi. Projector pada checkpoint sengaja tidak digunakan. Toleransi probabilitas 1e-5 dan prediksi kelas harus identik; kegagalan menghentikan test.

In [ ]:
def restore(run):
    keras.backend.clear_session();gc.collect()
    m=build_model(run['student'],512,42,weights=None,stage='head')
    status=weight_snapshot(m).restore(run['checkpoint'])
    status.assert_existing_objects_matched()
    status.expect_partial() # checkpoint contains training-only projector
    return m
def predict(m,run,split):
    return m.predict(make_dataset(FRAMES[split],IMAGE_DIR,run['student'],8,42,size=512),verbose=0).reshape(-1)
RESTORE_PASSED=False;rows=[]
unchanged()
for run in RUNS:
    m=restore(run)
    saved=align(pd.read_csv(run['folder']/'validation_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip'),FRAMES['validation'])
    p=predict(m,run,'validation');q=saved.probability.to_numpy(float)
    passed=bool(np.isfinite(p).all() and np.allclose(p,q,atol=1e-5,rtol=1e-5) and np.array_equal(p>=.5,q>=.5))
    rows.append(dict(student=run['student'],teacher=run['teacher'],passed=passed,max_abs_difference=float(np.max(abs(p-q))),class_disagreements=int(np.sum((p>=.5)!=(q>=.5)))))
    pd.DataFrame(rows).to_csv(OUT/'validation_restore_audit.csv',index=False)
    del m;keras.backend.clear_session();gc.collect()
display(pd.DataFrame(rows))
if not all(r['passed'] for r in rows):raise RuntimeError('Restore failed; test not evaluated. Kirim audit, jangan longgarkan toleransi.')
RESTORE_PASSED=True

## Cell 5 — Evaluasi test dan perbandingan control B06B

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
RESULTS=[]
if EVALUATE_TEST:
    if not RESTORE_PASSED:raise RuntimeError('Complete validation restore audit first')
    unchanged()
    for run in RUNS:
        dest=OUT/(run['student']+'__from__'+run['teacher']);dest.mkdir(exist_ok=True)
        donepath=dest/'EVALUATION_COMPLETE.json'
        if donepath.exists():
            done=read_json(donepath)
            if done['lock_sha256']!=objsha(LOCK):raise RuntimeError('Test result lock mismatch')
            for name,h in done['output_sha256'].items():
                if sha(dest/name)!=h:raise RuntimeError('Test output changed')
            result=read_json(dest/'test_metrics.json')
            table=align(pd.read_csv(dest/'test_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip'),FRAMES['test'])
            for k,v in metrics(table.label,table.probability).items():
                if not np.isclose(v,result[k],atol=1e-12):raise RuntimeError('Saved metrics mismatch')
        else:
            print('TEST',DATASET,run['student'],'<-',run['teacher'],flush=True)
            m=restore(run);p=predict(m,run,'test');del m;keras.backend.clear_session();gc.collect()
            result=dict(metrics(FRAMES['test'].label,p),model=run['student'],teacher=run['teacher'],dataset=DATASET,
                training_seed=42,split='test',method='supervised_control' if run['teacher']=='none' else 'ssid_inspired_feature_kd',
                lambda_feature=run['config']['lambda_feature'],source_run=str(run['folder']))
            table=FRAMES['test'][['sample_id','patient_id','label']].copy();table['probability']=p.astype(float);table['prediction']=(p>=.5).astype(int)
            table.to_csv(dest/'test_predictions.csv',index=False);save(dest/'test_metrics.json',result)
            names=['BI-RADS 1/2','BI-RADS 5/6'] if DATASET=='inbreast' else ['Benign','Malignant']
            fig,ax=plt.subplots(figsize=(5.6,5))
            ConfusionMatrixDisplay(np.array([[result['tn'],result['fp']],[result['fn'],result['tp']]]),display_labels=names).plot(ax=ax,cmap='Blues',values_format='d',im_kw={'vmin':0,'vmax':int(FRAMES['test'].label.value_counts().max())})
            ax.set_title(f"{run['student']} <- {run['teacher']}\n{DATASET} test; seed42; threshold0.5",fontsize=9)
            fig.tight_layout();fig.savefig(dest/'test_confusion_matrix.png',dpi=180);plt.close(fig)
            names=['test_predictions.csv','test_metrics.json','test_confusion_matrix.png']
            save(donepath,dict(lock_sha256=objsha(LOCK),output_sha256={n:sha(dest/n) for n in names}))
        RESULTS.append(result)
        pd.DataFrame(RESULTS).to_csv(OUT/'test_summary.csv',index=False)
    df=pd.DataFrame(RESULTS);display(df[['model','teacher','precision','recall','f1','macro_f1','specificity','accuracy','roc_auc']].round(4))
    deltas=[]
    for r in RESULTS:
        if r['teacher']=='none':continue
        control=df[(df.model==r['model'])&(df.teacher=='none')].iloc[0]
        deltas.append(dict(model=r['model'],teacher=r['teacher'],dataset=DATASET,**{'delta_'+k:r[k]-float(control[k]) for k in ['precision','recall','f1','macro_f1','specificity','accuracy','balanced_accuracy','roc_auc','average_precision']}))
    pd.DataFrame(deltas).to_csv(OUT/'test_kd_vs_control.csv',index=False);display(pd.DataFrame(deltas).round(4))
    unchanged();print('Selesai enam run test. Tidak ada training ulang.')
else:print('Audit-only: test belum dievaluasi.')

## Cell 6 — ZIP laporan
Dapat dijalankan setelah error audit untuk mengirim laporan parsial. File bobot tetap di B06B. Ulangi Cell2–6 untuk dataset berikutnya; Cell1 cukup sekali dalam runtime yang sama.

In [ ]:
REPORT=OUT/f'B06C_{DATASET}_seed42_test_report.zip'
save(OUT/'report_status.json',dict(completed_test_runs=len(list(OUT.glob('*/EVALUATION_COMPLETE.json'))),expected_runs=6,
    restore_passed=bool(globals().get('RESTORE_PASSED',False)),test_enabled=EVALUATE_TEST))
(OUT/'b06c_evaluator.py').write_text(EVALUATOR_SOURCE)
with zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ['.json','.csv','.png','.py']:z.write(p,str(p.relative_to(OUT)))
print(REPORT)
if DOWNLOAD_REPORT:files.download(str(REPORT))